## Integración con Langchain y langgraph

## Infra

In [1]:
from dotenv import load_dotenv
import os

import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

from openai import OpenAI

import ast

from IPython.display import HTML, Markdown, display
import pandas as pd

import tiktoken
import tqdm


# Add this line to load variables from .env file
load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')  # Retrieve the API key
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')  # Retrieve the API key
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')  # Retrieve the API key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

mod_alto = 'gpt-4.1-2025-04-14' # toma ~30 segs para responder correctamente
mod_bajo = 'gpt-4.1-nano-2025-04-14' # no puede
mod_med = 'gpt-4.1-mini-2025-04-14' # no puede


client = OpenAI()

client.embeddings.create(
  model='text-embedding-3-large', #"text-embedding-ada-002",
  input="The food was delicious and the waiter...",
  encoding_format="float"
)

embedding_fun_openai = OpenAIEmbeddingFunction(api_key=os.environ.get('OPENAI_API_KEY'), model_name="text-embedding-ada-002")

In [2]:
! git branch

* db_f1
  master
  test_enc_1
  test_enc_2


In [3]:
# Install and configure ChromaDB with DuckDB+Parquet backend
from chromadb.config import Settings  
from chromadb import PersistentClient                             # CHANGED
from chromadb.config import Settings, DEFAULT_TENANT, DEFAULT_DATABASE  # CHANGED

# Reinitialize Chroma client with DuckDB+Parquet backend (new ChromaDB API)
from chromadb import Client
from chromadb.config import Settings

DB_PERSIST_DIR = "./db_f1"
DB_NAME        = "enc_dbf1"


client = PersistentClient(
    path     = DB_PERSIST_DIR,
    settings = Settings(allow_reset=True),   # allow resetting on-disk state
    tenant   = DEFAULT_TENANT,
    database = DEFAULT_DATABASE,
) 

print("Collections:", [c.name for c in client.list_collections()])  

db_f1 = client.get_or_create_collection(
    name               = DB_NAME,
    embedding_function = embedding_fun_openai
)

db_f1.peek()

# db_f1 = client.get_collection(name="enc_dbf1")

Collections: ['enc_dbf1']


{'ids': ['p1_1a_1|IDE__question',
  'p1_1a_1|IDE__summary',
  'p1_1a_1|IDE__implications',
  'p2_1a_1|IDE__question',
  'p2_1a_1|IDE__summary',
  'p2_1a_1|IDE__implications',
  'p3_1|IDE__question',
  'p3_1|IDE__summary',
  'p3_1|IDE__implications',
  'p3_2|IDE__question'],
 'embeddings': array([[-0.01286233, -0.02477617,  0.00845953, ..., -0.00216455,
         -0.01462473, -0.00672276],
        [-0.00771893,  0.00366222,  0.03773989, ..., -0.00310992,
         -0.00857366, -0.0436573 ],
        [-0.02147629,  0.00293677,  0.03042583, ...,  0.00184429,
         -0.02441142, -0.04355532],
        ...,
        [ 0.01838034,  0.00962718,  0.02427666, ..., -0.00525712,
         -0.02070235, -0.05458009],
        [-0.00535758,  0.0059622 ,  0.01842068, ..., -0.00686241,
         -0.02355321, -0.02821548],
        [-0.00732108, -0.0118375 ,  0.03220749, ..., -0.00837447,
         -0.01164658, -0.00516163]], shape=(10, 1536)),
 'documents': ['IDENTIDAD_Y_VALORES|Con la palabra maíz, yo asocio

In [4]:
# clientes para encoding de query, etc

client = OpenAI()

client.embeddings.create(
  model='text-embedding-3-large', #"text-embedding-ada-002",
  input="The food was delicious and the waiter...",
  encoding_format="float"
)

embedding_fun_openai = OpenAIEmbeddingFunction(api_key=os.environ.get('OPENAI_API_KEY'), model_name="text-embedding-ada-002")

In [5]:
# importar pickle con datos de Los Mexicanos

import pickle

ruta_enc= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/encuestas/'
ruta_rep= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/reportes/'
ruta_tmp_images= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/tmp_images/'

with open(f'{ruta_enc}/los_mex_dict.pkl', 'rb') as f:
    los_mex_dict = pickle.load(f)
    print('los_mex_dict cargado --  leer readme_dict para info')


enc_dict = los_mex_dict['enc_dict']
enc_nom_dict = los_mex_dict['enc_nom_dict']
rev_enc_nom_dict = {v: k for k, v in enc_nom_dict.items()}

pregs_dict = los_mex_dict['pregs_dict']
ses_dict = los_mex_dict['ses_dict']
mkdown_tables = los_mex_dict['mkdown_tables']
df_tables = los_mex_dict['df_tables']

# TODO: usar df_tables para plots

los_mex_dict cargado --  leer readme_dict para info


In [6]:
len(enc_nom_dict)

26

In [7]:
rev_topic_dict = {k: v.replace('_', ' ').lower() for k, v in rev_enc_nom_dict.items()}
topic_id_st = '\n'.join(['|'.join(['* ' + a, b]) for a, b in rev_topic_dict.items()])

### funciones utilitarias

In [8]:
# contador de tokens y batcher 


def num_tokens_from_string(string: str, encoding_name: str) -> int:
    # Utility function to calculate the number of tokens in a string.
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Update the token count calculation in batch_documents to use num_tokens_from_string
# 8192 is the token limit for the model
def batch_documents(docs, ids, max_tokens=8192, encoding_name="cl100k_base"):
    # This function batches documents and their IDs while respecting the token limit.
    # It ensures that each batch does not exceed the maximum token limit,
    # which is crucial for efficient processing with the LLM.
    batches = []
    current_batch_docs = []
    current_batch_ids = []
    current_token_count = 0

    for doc, doc_id in zip(docs, ids):
        doc_token_count = num_tokens_from_string(doc, encoding_name)  # Use the token counting function
        
        # print(f"Document ID: {doc_id}, Token Count: {doc_token_count}")
        
        if current_token_count + doc_token_count > max_tokens:
            # Save the current batch and start a new one
            batches.append((current_batch_docs, current_batch_ids))
            current_batch_docs = []
            current_batch_ids = []
            current_token_count = 0

        # Add the document to the current batch
        current_batch_docs.append(doc)
        current_batch_ids.append(doc_id)
        current_token_count += doc_token_count

    # Add the last batch if it has any documents
    if current_batch_docs:
        batches.append((current_batch_docs, current_batch_ids))

    return batches

In [9]:
# llamdas a LLM y limpieza de salida a json
def get_answer(prompt, system_prompt=None, model='gpt-4o-mini-2024-07-18', temperature=0.9):
    # This function sends the prompt to the LLM and retrieves the response.
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user",   "content": prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


def clean_llm_json_output(content):
    # Remove code block markers and any leading/trailing text
    content = content.strip()
    # Remove code block markers
    content = re.sub(r"^```json", "", content, flags=re.IGNORECASE).strip()
    content = re.sub(r"^```", "", content, flags=re.IGNORECASE).strip()
    content = re.sub(r"```$", "", content, flags=re.IGNORECASE).strip()
    # Remove any leading text before the first {
    idx = content.find('{')
    if idx > 0:
        content = content[idx:]
    # Remove trailing text after last }
    idx2 = content.rfind('}')
    if idx2 > 0:
        content = content[:idx2+1]
    return content


## query

### query v2: creación de resúmenes temáticos

In [10]:
# query de la base de embeddings
# query_emb es el embedding de la pregunta

user_query= '¿qué significa ser mexicano para los mexicanos?' 
#'¿cómo usan su tipempo los mexicanos?'

#'qué tanto le importa la niñez a los mexicanos?'
# turn it into a vector
query_emb = embedding_fun_openai([user_query])[0]
query_emb

array([-0.01175261, -0.02690859,  0.01918369, ...,  0.00272391,
       -0.02097107, -0.0183757 ], shape=(1536,), dtype=float32)

## content_describer

In [11]:
# TODO - en desarrollo en dataset_knowledge.py

## variable_selector

In [12]:
# ejemplo: query entre las preguntas

result_q = db_f1.query(
    query_embeddings = [query_emb],
    n_results        = 5,
    where            = {"type": "question"}
)

print(result_q["documents"])   # the matched question strings
print(result_q["metadatas"])   # metadata for each hit (includes qid & type)


[['CULTURA_POLITICA|¿Qué tan orgulloso se siente de ser mexicano?', 'IDENTIDAD_Y_VALORES|¿Qué tan orgulloso se siente de ser mexicano?', 'GLOBALIZACION|¿Qué tan orgulloso está de ser mexicano?', 'FEDERALISMO|¿Qué tan orgulloso se siente usted de ser mexicano?', 'INDIGENAS|¿Cuál cree que es la mayor ventaja de ser indígena en México?']]
[[{'qid': 'p5|CUL', 'type': 'question'}, {'type': 'question', 'qid': 'p6|IDE'}, {'type': 'question', 'qid': 'p15|GLO'}, {'type': 'question', 'qid': 'p19_3|FED'}, {'qid': 'p13|IND', 'type': 'question'}]]


In [13]:
# tmp_dist_df contiene las distancias para los tres tipos/ facets, normalizadas entre 0 y 1

# OJO: esto obviamente asume que los tres tipos de información tienen la misma importancia, lo cual no es inicialmente cierto.
# Pero la mezcla de las tres calificaciones devuelve una variedad más amplia de resultados.  

type_lst = [ "question", "summary", "implications"]

tmp_dist_dict = {}
for type in type_lst:
    print(f"Querying for type: {type}")
    tmp_res_q = db_f1.query(
        query_embeddings = [query_emb],
        n_results        = 100,  # devuelvo 100 resultados para cada tipo con distancias menores
        where            = {"type": type}
    )
    [tmp_res_ids] = tmp_res_q['ids']
    [tmp_res_distances] = tmp_res_q['distances']

    tmp_dist_dict[type]= dict(zip(tmp_res_ids, tmp_res_distances))

# remove the suffixes from the keys
tmp_dist_dict = { outer_key: { k.split('__')[0]: v for k, v in inner_dict.items() }
    for outer_key, inner_dict in tmp_dist_dict.items() }

# Create a DataFrame where keys in every subdict are the index and keys in tmp_dist_dict are columns
tmp_dist_df = pd.DataFrame.from_dict(tmp_dist_dict)

# Normalize each column so that max = 1 and min = 0
tmp_dist_df = (tmp_dist_df - tmp_dist_df.min()) / (tmp_dist_df.max() - tmp_dist_df.min())

Querying for type: question
Querying for type: summary
Querying for type: implications


In [14]:
# tmp_res_st contiene las n preguntas más cercanas y la descripción de sus resultados.

top_vals = 30

tmp_dist_df['mean'] = tmp_dist_df.mean(axis=1)
tmp_dist_df.sort_values(by='mean', ascending=True, inplace=True)

top_ids = tmp_dist_df.head(top_vals).index.tolist()

tmp_list = []

for type in type_lst:
    for id in top_ids:
        tmp_list.append(id + f'__{type}')


# Retrieve documents using the list of ids
result_by_ids = db_f1.get(ids=tmp_list)

tmp_pre_res_dict = dict(zip(result_by_ids['ids'], result_by_ids['documents']))

# tmp_preproc_dic = {k: v for k, v in tmp_pre_res_dict.items() if k.split('__')[1] in ['question', 'summary'] }

# # # instead of your current list-comp do this:
# tmp_combined_strings = []

#  # Calculate the group index based on the position

# for i, (k, v) in enumerate(tmp_preproc_dic.items(), start=1):
#     facet = k.split("__", 1)[1].upper()
#     p_id = k#.split("__")[0].split("|")[0]

#     grouped_index = (i + 1) // 2 
#     # only split once, and if there is no "|" take the entire v
#     q_id = v.split("|")[0]
#     parts = v.split("|", 1)
#     text = parts[1] if len(parts) > 1 else parts[0]

#     p_id = p_id.split("__")[0]

#     tmp_combined_strings.append(f"{facet}_{grouped_index}|{p_id}: {text}")

# tmp_res_st = '\n'.join(tmp_combined_strings)
# tmp_res_st

In [15]:


# tmp_SEL_lst = list(set([st.split('__')[0] for st in tmp_preproc_dic.keys()]))
# tmp_RES_lst= []
# for ky_st in tmp_SEL_lst:
#     tmp_var_st = ky_st
#     tmp_dict = {k:v for k,v in tmp_preproc_dic.items() if k.startswith(tmp_var_st)}
#     tmp_RES_st = '\n'.join([f' * QUESTION_ID: {tmp_var_st}', 
#                             f' QUESTION: {tmp_dict[tmp_var_st+"__question"].split("|")[1]}',
#                             f' SUMMARY: {tmp_dict[tmp_var_st+"__summary"]}', 
#                         ])
#     tmp_RES_lst.append(tmp_RES_st)

# tmp_PRMT_st= '\n'.join(tmp_RES_lst)
# print(tmp_PRMT_st)

### filtro de relevancia

In [16]:
## generación del esquema de pydantic para la respuesta

from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

# 1. Define Pydantic model for an entry
class PatternItemGrader(BaseModel):
    GRADE_DICT: dict[float, str] = {
        0.0: "",
    }

# 2. Create parser for single entries
pattern_parser_grader = PydanticOutputParser(pydantic_object=PatternItemGrader)
pattern_format_grader_instrtuctions = pattern_parser_grader.get_format_instructions()

In [17]:
def create_prompt_grader(user_query, tmp_svvinfo_st ,format_instructions=""):
    """
    prompt for gradig items against a query
    """
    prompt = f"""
You are an expert in survey research and in qualitative research, and you are fluent in English and Spanish. You will reply in English only.  
Your task is to read the following SURVEY INFORMATION, and a user QUERY, and then grade the SURVEY INFORMATION against the QUERY and write a one-sentence explanation of your grade.

THE SURVEY INFORMATION has 3 parts
- QUESTION: the question asked in the survey
- SUMMARY: a summary of the results of the survey
- IMPLICATIONS: a statement of the importance of the results written by an expert in the field of the survey. 

The GRADE will will be a number between 0 and 3, where: 
- 0: the QUESTION and the SUMMARY are NOT relevant to the QUERY, and the IMPLICATIONS are NOT relevant to the QUERY
- 1: the QUESTION and the SUMMARY are NOT relevant to the QUERY, but the IMPLICATIONS seem relevant to the QUERY
- 2: the QUESTION and the SUMMARY are somewhat relevant to the QUERY, but the IMPLICATIONS seem relevant to the QUERY
- 3: the QUESTION and the SUMMARY are relevant to the QUERY, and the IMPLICATIONS are relevant to the QUERY

You will write a one-sentence EXPLANATION of your grade, paying attention to how QUESTION, SUMMARY and IMPLICATIONS are related to the QUERY. Be detailed and specific in your explanation.
IMPORTANT: return an EXPLANATION regardless of the GRADE, this is, explain all your grades, even if they are 0.
IMPORTANT: make sure to match your GRADE to your EXPLANATION, this is, that they correspond to the criteria above. 

Here are some examples of the GRADE and the explanation:
- EXAMPLE_QUERY : "Would selling strawberry ice cream is a good idea?"

- EXAMPLE_SURVEY_INFORMATION_1 : "QUESTION: Do you like strawberry ice cream? SUMMARY: 50% of people like strawberry ice cream. IMPLICATIONS: Selling strawberry ice cream is a good idea."
- EXAMPLE_GRADE_1 : 3
- EXAMPLE_EXPLANATION_1 : "The QUESTION is relevant to the QUERY because it asks about strawberry ice cream, the SUMMARY is relevant because it provides information about people's preferences, and the IMPLICATIONS are relevant because they suggest that selling strawberry ice cream is a good idea."

- EXAMPLE_SURVEY_INFORMATION_2 : "QUESTION: Do you like chocolate ice cream? SUMMARY: 50% of people like chocolate ice cream. IMPLICATIONS: Selling chocolate ice cream is a good idea."  
- EXAMPLE_GRADE_2 : 2
- EXAMPLE_EXPLANATION_2 : "The QUESTION is not relevant to the QUERY because it asks about chocolate ice cream, but the SUMMARY is relevant because it provides information about people's preferences, and the IMPLICATIONS are relevant because they suggest that selling chocolate ice cream is a good idea for alternate flavors."

- EXAMPLE_SURVEY_INFORMATION_3 : "QUESTION: Have you seen the movie 'Wild Strawberries' by Ingmar Bergman? SUMMARY: 50% of people have seen the movie. IMPLICATIONS: The movie is a classic and is worth watching."
- EXAMPLE_GRADE_3 : 1
- EXAMPLE_EXPLANATION_3 : "The QUESTION is not relevant to the QUERY because it asks about a movie, and the SUMMARY is not relevant because it talks about movies, not ice cream, and the IMPLICATIONS are seem relevant because they talk about a movie with 'ice cream' in the title."

- EXAMPLE_SURVEY_INFORMATION_4 : "QUESTION: How many times have you gone to the beach? SUMMARY: 50% of people go to the beach once a year. IMPLICATIONS: The beach is a popular destination."
- EXAMPLE_GRADE_4 : 0
- EXAMPLE_EXPLANATION_4 : "The QUESTION is not relevant to the QUERY because it asks about going to the beach, and the SUMMARY is not relevant because it talks about the beach, not ice cream, and the IMPLICATIONS are not relevant because they talk about a beach, not ice cream."

Example output (strict JSON, no markdown, no code block, no extra text).
{{
  GRADE: EXPLANATION
}}

QUERY: {user_query}
SURVEY_INFORMATION: {tmp_svvinfo_st}

{format_instructions}

Checklist before submitting:
- [ ] GRADE has been calculated.
- [ ] EXPLANATION has been written.
- [ ] No field is left empty.
"""
    return prompt
prompt = create_prompt_grader(user_query, tmp_svvinfo_st= "" ,format_instructions="")

print(prompt)


You are an expert in survey research and in qualitative research, and you are fluent in English and Spanish. You will reply in English only.  
Your task is to read the following SURVEY INFORMATION, and a user QUERY, and then grade the SURVEY INFORMATION against the QUERY and write a one-sentence explanation of your grade.

THE SURVEY INFORMATION has 3 parts
- QUESTION: the question asked in the survey
- SUMMARY: a summary of the results of the survey
- IMPLICATIONS: a statement of the importance of the results written by an expert in the field of the survey. 

The GRADE will will be a number between 0 and 3, where: 
- 0: the QUESTION and the SUMMARY are NOT relevant to the QUERY, and the IMPLICATIONS are NOT relevant to the QUERY
- 1: the QUESTION and the SUMMARY are NOT relevant to the QUERY, but the IMPLICATIONS seem relevant to the QUERY
- 2: the QUESTION and the SUMMARY are somewhat relevant to the QUERY, but the IMPLICATIONS seem relevant to the QUERY
- 3: the QUESTION and the SU

In [18]:
import tiktoken
import re
import json


def get_structured_summary_grader(tmp_svvinfo_st, model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    prompt =create_prompt_grader(user_query=user_query, tmp_svvinfo_st= tmp_svvinfo_st, format_instructions=pattern_format_grader_instrtuctions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_grader.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_grader.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}



tmp_ky = 15
tmp_id_st = top_ids[tmp_ky]
tmp_svyinfo_dict = {k.split('__')[1].upper(): v for k,v in tmp_pre_res_dict.items() if k.startswith(tmp_id_st)}
tmp_svvinfo_st = ' '.join([f'{k}: {v}' for k,v in tmp_svyinfo_dict.items()])


content, tmp_grade_dict = get_structured_summary_grader(tmp_svvinfo_st, model_name = mod_bajo, temperature= 0.1)
print(content)

# TODO: batch queries, review grades, test new queries and models. 


{"GRADE_DICT": {"3": "The QUESTION directly addresses the meaning of being Mexican, which is relevant to the QUERY; the SUMMARY provides detailed insights into respondents' perceptions of their Mexican and Yucateco identities, which is relevant; and the IMPLICATIONS highlight the importance of understanding identity complexities for cultural and social applications, making all parts highly relevant."}}


In [19]:
def create_tmp_svyinfo_dict(tmp_ky, top_ids, tmp_pre_res_dict):
    """         
    Create a temporary survey information dictionary for a given key.
    Args:
        tmp_ky (int): The index of the key in the top_ids list.
        top_ids (list): The list of top IDs.
        tmp_pre_res_dict (dict): The dictionary containing preprocessed results.
    Returns:

        dict: A dictionary containing the survey information for the specified key.
    """
    # Create a temporary survey information dictionary for a given key
    tmp_id_st = top_ids[tmp_ky]
    tmp_svyinfo_dict = {k.split('__')[1].upper(): v for k,v in tmp_pre_res_dict.items() if k.startswith(tmp_id_st)}
    tmp_svvinfo_st = ' '.join([f'{k}: {v}' for k,v in tmp_svyinfo_dict.items()])
    return tmp_svvinfo_st

# Example usage
tmp_ky = 1
tmp_svyinfo_dict = create_tmp_svyinfo_dict(tmp_ky, top_ids, tmp_pre_res_dict)
print(tmp_svyinfo_dict)

QUESTION: IDENTIDAD_Y_VALORES|¿Qué tan orgulloso se siente de ser mexicano? SUMMARY: The table presents data on how proud individuals feel about being Mexican, with 63.42% expressing a strong sense of pride ('Mucho'), while only 25.42% feel some pride ('Algo'). A small percentage feel little pride ('Poco') or no pride at all ('Nada'), and 0.50% do not identify as Mexican. Notably, 0.25% of respondents did not provide an answer, which is below the 2.0% threshold, indicating that most respondents felt comfortable expressing their views. IMPLICATIONS: This table is vital for experts in 'identidad y valores' as it provides quantitative insight into national identity and the emotional connection individuals have with their nationality. Such data can inform policies, educational programs, and community initiatives aimed at fostering pride and belonging. Others can leverage these insights to understand cultural trends, enhance marketing strategies, or develop social programs that resonate wit

In [20]:
def get_structured_summary_grader_p(prompt, model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    # prompt =create_prompt_grader(user_query=user_query, tmp_svvinfo_st= tmp_svvinfo_st, format_instructions=pattern_format_grader_instrtuctions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_grader.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_grader.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}
    
def create_tmp_svyinfo_dict(tmp_ky, top_ids, tmp_pre_res_dict):
    """         
    Create a temporary survey information dictionary for a given key.
    Args:
        tmp_ky (int): The index of the key in the top_ids list.
        top_ids (list): The list of top IDs.
        tmp_pre_res_dict (dict): The dictionary containing preprocessed results.
    Returns:

        dict: A dictionary containing the survey information for the specified key.
    """
    # Create a temporary survey information dictionary for a given key
    tmp_id_st = top_ids[tmp_ky]
    tmp_svyinfo_dict = {k.split('__')[1].upper(): v for k,v in tmp_pre_res_dict.items() if k.startswith(tmp_id_st)}
    tmp_svvinfo_st = ' '.join([f'{k}: {v}' for k,v in tmp_svyinfo_dict.items()])
    return tmp_svvinfo_st

def batch_process_expert_grader(user_query, top_ids, tmp_pre_res_dict, model_name , batch_size=8192):
    """
    Processes expert summaries in batches, saving checkpoints after each batch.

    Parameters:
        top_ids (list): List of variables to grade.
        tmp_pre_res_dict (dict): Dict with survey results.
        batch_size (int): Maximum token limit for each batch.

    Returns:
        dict: A dictionary of structured results.
    """


    # Prepare prompts and keys
    prompts = [
        create_prompt_grader(user_query,  
                             create_tmp_svyinfo_dict(tmp_ky, top_ids, 
                                                     tmp_pre_res_dict),
                                                     format_instructions=pattern_format_grader_instrtuctions)
        for tmp_ky in range(len(top_ids))
    ]
    keys = top_ids
    
    # Batch the documents
    batches = batch_documents(prompts, keys, max_tokens=batch_size, encoding_name="cl100k_base")
    
    # Initialize results dictionary
    structured_results = {}
    
    # Process each batch
    with tqdm.tqdm(total=len(keys), desc="Processing Expert Summaries") as pbar:
        for batch_docs, batch_keys in batches:
            for prompt, key in zip(batch_docs, batch_keys):
                try:
                    _, tmp_grade_dict = get_structured_summary_grader_p(prompt, model_name = model_name, temperature= 0.5)
                    #print(f'tmp_grade_dict: {tmp_grade_dict}')
                    structured_results[key] = tmp_grade_dict
                except Exception as e:
                    structured_results[key] = {'error': str(e)}
                pbar.update(1)
    
    return structured_results

tst_res = batch_process_expert_grader(user_query, top_ids, tmp_pre_res_dict, mod_alto, batch_size=8192)
tmp_grade_dict=  {k: v['GRADE_DICT'] for k, v in tst_res.items()}

# Filtrar los elementos que tienen una calificación mayor a 1

tmp_grade_dict= {k: v for k, v in tmp_grade_dict.items() if list(v.keys())[0] >1 }
tmp_grade_dict

Processing Expert Summaries: 100%|██████████| 30/30 [00:59<00:00,  1.97s/it]


{'p5|CUL': {3.0: 'The QUESTION is relevant to the QUERY because it asks about how proud people feel to be Mexican, which is directly related to what it means to be Mexican for Mexicans; the SUMMARY is relevant because it provides quantitative data on these feelings of pride; and the IMPLICATIONS are relevant because they discuss how this sense of pride and identity can inform understandings of national identity and related policies.'},
 'p6|IDE': {3.0: 'The QUESTION directly addresses Mexican identity and pride, which is highly relevant to the QUERY about what it means to be Mexican; the SUMMARY provides detailed quantitative data on pride in being Mexican, and the IMPLICATIONS discuss the importance of this information for understanding national identity, making all three components directly relevant to the QUERY.'},
 'p1_1a_1|IDE': {3.0: "The QUESTION is relevant to the QUERY because it directly asks about associations with 'México', which relates to what being Mexican means to Mexic

In [21]:
tmp_grade_dict

{'p5|CUL': {3.0: 'The QUESTION is relevant to the QUERY because it asks about how proud people feel to be Mexican, which is directly related to what it means to be Mexican for Mexicans; the SUMMARY is relevant because it provides quantitative data on these feelings of pride; and the IMPLICATIONS are relevant because they discuss how this sense of pride and identity can inform understandings of national identity and related policies.'},
 'p6|IDE': {3.0: 'The QUESTION directly addresses Mexican identity and pride, which is highly relevant to the QUERY about what it means to be Mexican; the SUMMARY provides detailed quantitative data on pride in being Mexican, and the IMPLICATIONS discuss the importance of this information for understanding national identity, making all three components directly relevant to the QUERY.'},
 'p1_1a_1|IDE': {3.0: "The QUESTION is relevant to the QUERY because it directly asks about associations with 'México', which relates to what being Mexican means to Mexic

## report_generator

In [57]:
# tmp_res_st es el string con las variables filtradas

tmp_preproc_dic = {k: v for k, v in tmp_pre_res_dict.items() if k.split('__')[1] in ['question', 'summary'] }

# filtro de las preguntas relevantes
tmp_preproc_dic ={k: v for k, v in tmp_preproc_dic.items() if any(k.startswith(grade_key) for grade_key in tmp_grade_dict.keys())}

tmp_combined_strings = []

for i, (k, v) in enumerate(tmp_preproc_dic.items(), start=1):
    facet = k.split("__", 1)[1].upper()
    p_id = k#.split("__")[0].split("|")[0]

    grouped_index = (i + 1) // 2 
    # only split once, and if there is no "|" take the entire v
    q_id = v.split("|")[0]
    parts = v.split("|", 1)
    text = parts[1] if len(parts) > 1 else parts[0]

    p_id = p_id.split("__")[0]

    tmp_combined_strings.append(f"{facet}_{grouped_index}|{p_id}: {text}")

tmp_res_st = '\n'.join(tmp_combined_strings)
tmp_res_st

"QUESTION_1|p1_1a_1|IDE: Con la palabra maíz, yo asocio comida, mercado, animales. Dígame por favor, tres palabras que asocies con la palabra MÉXICO. 1° MENCIÓN\nSUMMARY_1|p1_1a_1|IDE: The table presents various associations that people have with the word 'México', showcasing a wide range of responses. The most prominent association is 'Orgullo', which reflects a strong sense of national pride at 25.25%, followed by 'Costumbres y tradiciones' at 13.50%, and 'Historia y cultura' at 7.58%. Notably, the 'No sabe/ No contesta' response is at 9.75%, indicating a significant portion of respondents either do not have an association or are unwilling to share. This highlights potential gaps in connection or sentiment towards the national identity.\nQUESTION_2|p2_1a_1|IDE: Y ahora, voy a pedir que me diga, por favor, tres palabras que asocie con la palabra MEXICANO. 1° MENCIÓN\nSUMMARY_2|p2_1a_1|IDE: The table presents various words and phrases associated with the identity of 'Mexicano', indicat

In [92]:

# objetos para el parser y formato de salida
class PatternVarRevAction(BaseModel):
    var_review_action_dict: dict[str, str] = {
        "ACTION": "",
        "QUESTION_IDS": "",
        "USER_QUERY": ""
    }

pattern_parser_VarRevAction = PydanticOutputParser(pydantic_object=PatternVarRevAction)
pattern_format_grader_VarRevAction = pattern_parser_VarRevAction.get_format_instructions()

# prompt
def create_prompt_var_reviewer(user_query, user_feedback, tmp_res_st ,format_instructions=""):
    """
    prompt for selecting items against a query
    """
    prompt = f"""
You are an expert survey research analyst. Your task is to review a list of survey questions that have been selected based on a user query and modify it according to the USER FEEDBACK. 
You will read a list of question IDs and their associated questions, as well as the USER FEEDBACK. 
Note that the question IDs have format 'question_id|table_id: question statement' (e.g., pxx|YYY where xx is any combination of numbers and '_', and YYY are any three capital letters). 

To modify the list, you have to read the USER FEEDBACK and choose one of the following actions:
1. ADD_QUERY: if the user asks to add questions by topic or area of interest, but not the specific question IDs, you will summarize the user original query and the new feedback in the langugue of the query, and retrun a single phrase to query the vector database.
    For example: if the user QUERY is "do people like strawberry ice cream?" and the SELECTED_QUESTION contains item 'p01|ABC: Do you like strawberry ice cream?', and the user USER FEEDBACK is "add questions about chocolate ice cream", you will return the USER_QUERY as "do people like strawberry or chocolate ice cream?".
    You will return field SELECTED QUESTIONS as an emtpy list. 
2. REMOVE_QUERY: if the user asks to remove questions by topic or area of interest, but not the specific question IDs, you will reply in the USER_QUERY that you cannot remove question by topic, and recommend the user reset the query.
3. ADD_QUESTION: if the user asks to add specific question IDs,  you will return them in the QUESTION_IDS field. If the user requested any variable that is not in the current selection, you will return the closest matching question IDs based on the current selection.  
4. REMOVE_QUESTION: if the user asks to remove specific question IDs, you will return them in the QUESTION_IDS field. If the user requested any variable that is not in the current selection, you will return the closest matching question IDs based on the current selection.
6. NO_CHANGE: if the user feedback is not applicable or does not require any changes to the current selection.

You will retun a JSON object with the following fields:
json
{{
  "ACTION": "ADD_QUERY" | "ADD_QUESTION" | "REMOVE_QUERY" | "REMOVE_QUESTION" | "RESTART_QUERY",
  "QUESTION_IDS": "the question IDs that the user mentioned to add or remove, separated by commas, or an empty string if not applicable",
  "USER_QUERY": 'the new information that the user requested to add to the variable query, or an empty string if not applicable',
}}

QUERY: {user_query}             
USER FEEDBACK: {user_feedback}
SELECTED QUESTIONS: {tmp_res_st}
{format_instructions}
"""
    return prompt

user_feedback = "quita las preguntas sobre economía"

# llamada al LLM para revisar variables
prompt= create_prompt_var_reviewer(
    user_query=user_query, 
    user_feedback=user_feedback, 
    tmp_res_st=tmp_res_st, 
    format_instructions=pattern_format_grader_VarRevAction
)


content = get_answer(prompt, model=mod_med, temperature=0.0)
content = clean_llm_json_output(content)
parsed = pattern_parser_VarRevAction.parse(content)

tst= pattern_parser_VarRevAction.parse(content)
var_review_action_dict = tst.model_dump()["var_review_action_dict"]
var_review_action_dict





{'ACTION': 'REMOVE_QUERY',
 'QUESTION_IDS': '',
 'USER_QUERY': 'No es posible quitar preguntas por tema, se recomienda reiniciar la consulta para excluir preguntas sobre economía.'}

In [93]:
if var_review_action_dict["ACTION"] == "REMOVE_QUERY":
    print(f'{var_review_action_dict['USER_QUERY']}')

No es posible quitar preguntas por tema, se recomienda reiniciar la consulta para excluir preguntas sobre economía.


In [89]:
question_ids_to_remove = var_review_action_dict["QUESTION_IDS"].split(",")
print(f"Removing question IDs: {question_ids_to_remove}")


tmp_tst_pgs_dict = {k.split('__')[0]: v for k, v in tmp_pre_res_dict.items() if k.split('__')[1] in ['question'] }

tmp_pregs_drop_lst = []
tmp_pregs_noid_lst = []

for name in question_ids_to_remove:
    if name in tmp_tst_pgs_dict.keys():
        tmp_pregs_drop_lst.append(name)
    else:
        tmp_pregs_noid_lst.append(name)

tmp_pre_res_dict_out = {k: v for k, v in tmp_pre_res_dict.items() if k.split('__')[0] not in tmp_pregs_drop_lst}


print(f"Preguntas a eliminar: {tmp_pregs_drop_lst}")
print(f"Preguntas no encontradas: {tmp_pregs_noid_lst}")

Removing question IDs: ['p37_8|QUESTION_5']
Preguntas a eliminar: []
Preguntas no encontradas: ['p37_8|QUESTION_5']


{'p4|IDE__question': 'IDENTIDAD_Y_VALORES|¿Con cuál de estas dos frases está usted más de acuerdo?',
 'p4|IDE__summary': 'The table presents opinions on national identity and values among Mexicans. A majority (55.25%) agree that a great nation can be built despite having distinct cultures and values, while 38.42% believe similarity in culture and values is necessary. A small portion (4.00% combined) selected spontaneous options indicating no agreement or were undecided, suggesting some uncertainty in public sentiment about national cohesion.',
 'p4|IDE__implications': "This table is crucial for experts in 'identidad y valores' as it highlights prevailing attitudes towards cultural diversity and national identity among Mexicans. By understanding these perspectives, policymakers and social organizations can tailor their initiatives to promote inclusivity and social unity, leveraging the insights to foster dialogue and address the concerns reflected in the data.",
 'p5_1|IDE__question': '

{'p1_1a_1|IDE': 'IDENTIDAD_Y_VALORES|Con la palabra maíz, yo asocio comida, mercado, animales. Dígame por favor, tres palabras que asocies con la palabra MÉXICO. 1° MENCIÓN',
 'p2_1a_1|IDE': 'IDENTIDAD_Y_VALORES|Y ahora, voy a pedir que me diga, por favor, tres palabras que asocie con la palabra MEXICANO. 1° MENCIÓN',
 'p4|IDE': 'IDENTIDAD_Y_VALORES|¿Con cuál de estas dos frases está usted más de acuerdo?',
 'p5_1|IDE': 'IDENTIDAD_Y_VALORES|¿Cuál de las siguientes emociones refleja mejor lo que siente sobre México?  1° MENCIÓN',
 'p37_8|IDE': 'IDENTIDAD_Y_VALORES|Por lo que usted piensa, ¿el gobierno debería o no debería intervenir en las decisiones con respecto a la organización de las elecciones?',
 'p78|IDE': 'IDENTIDAD_Y_VALORES|¿Usted cree que el gobierno se debe apoyar en las ideas de la Revolución Mexicana o debe cambiar de ideas?',
 'p32|POB': 'POBREZA|En su opinión, ¿cree que en México hay mexicanos de primera y segunda o que todos son iguales?',
 'p5|CUL': 'CULTURA_POLITICA|¿

In [ ]:
# query de la base de embeddings
# query_emb es el embedding de la pregunta

user_query= '¿qué significa ser mexicano para los mexicanos?' 
#'¿cómo usan su tipempo los mexicanos?'

#'qué tanto le importa la niñez a los mexicanos?'
# turn it into a vector
query_emb = embedding_fun_openai([user_query])[0]
query_emb

### investigación de patrones lógicos (tst_lgc_dict)

In [ ]:
## generación del esquema de pydantic para la respuesta

from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser
from typing import Dict

# 1. Define Pydantic model for an entry

class PatternItem(BaseModel):
    TITLE_SUMMARY: str
    VARIABLE_STRING: str
    DESCRIPTION: str

class FlatPatternSummary(BaseModel):
    """
    A “catch-all” model so we can declare SIMILAR_PATTERN_1…n
    and DIFFERENT_PATTERN_1…n at the top level.
    """
    class Config:
        extra = "allow"


def flat_pattern_template(n: int) -> FlatPatternSummary:
    """
    Build an “empty” flat summary with n SIMILAR_PATTERN_i
    and n DIFFERENT_PATTERN_i placeholders.
    """
    empty = {"TITLE_SUMMARY": "", "VARIABLE_STRING": "", "DESCRIPTION": ""}
    payload: dict[str, dict[str, str]] = {}
    for kind in ("SIMILAR_PATTERN", "DIFFERENT_PATTERN"):
        for i in range(1, n + 1):
            payload[f"{kind}_{i}"] = empty.copy()
    return FlatPatternSummary(**payload)


# example: build a 3×3 skeleton and print its JSON
tmpl = flat_pattern_template(3)
# print(tmpl.model_dump_json(indent=2))

from langchain.output_parsers import PydanticOutputParser

# re-create your parser against the PatternSummaryEntry template
pattern_parser_simdif = PydanticOutputParser(pydantic_object=FlatPatternSummary)
pattern_simdif_format_instructions = pattern_parser_simdif.get_format_instructions()

# optional: inspect the format instructions
print(pattern_simdif_format_instructions)



The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"additionalProperties": true, "description": "A “catch-all” model so we can declare SIMILAR_PATTERN_1…n\nand DIFFERENT_PATTERN_1…n at the top level.", "properties": {}}
```


In [20]:
def create_prompt_crosssum(user_query, tmp_res_st, n_topics=5, format_instructions=""):
    """
    Optimized prompt for extracting non-empty, detailed patterns from survey results.
    """
    prompt = f"""
You are a research assistant analyzing survey results to answer the QUERY below.

Your job is to:
- Identify the top {n_topics} SIMILAR PATTERNS (trends or agreements) and {n_topics} DIFFERENT PATTERNS (contrasts or contradictions) relevant to the QUERY.
- For each pattern, provide:
    1. TITLE_SUMMARY: A short, descriptive title (never empty).
    2. VARIABLE_STRING: Comma-separated QUESTION_IDs used in the pattern (never empty).
    3. DESCRIPTION: A detailed explanation, citing numbers and QUESTION_IDs in parentheses (never empty).

Instructions:
- Do NOT leave any field empty. If information is limited, generalize or summarize what is available.
- Use only facts you are sure about, and always cite the QUESTION_ID for each fact.
- Do NOT repeat the same pattern in multiple fields.
- Ignore any results marked as 'NaN' or not available.
- If you combine categories (e.g., "like a lot" + "like somewhat"), only mention the sum if all original values are present.
- Do NOT invent data; if a pattern is weak, explain that.
- Each pattern must be unique and relevant to the QUERY.

Example input format:
* QUESTION_ID: p1_1|ABC
  QUESTION: 'Do you like ice cream?'
  SUMMARY: 85% of people like ice cream, 10% do not, 5% do not know.
* QUESTION_ID: p2_1|ABC
  QUESTION: 'Do you like marshmallow treats?'
  SUMMARY: 80% like, 15% do not, 5% do not know.
* QUESTION_ID: p10_1|DEF:
  QUESTION: 'Do you like sour apple candies?'
  SUMMARY: 40% of people like sour apple candies, while 50% said they do not like it, and 10% said they do not know.

In these cases, the SIMILAR PATTERN is that 'people really like sweet treats because 85% like ice cream and 80% like marshmallow treats', 
and the DIFFERENT PATTERN is that 'people do not like sour treats as much because only 40% said they liked them'.

Example output for a SIMILAR PATTERN:
TITLE_SUMMARY: High preference for sweet treats
VARIABLE_STRING: p1_1|ABC,p2_1|ABC
DESCRIPTION: A large majority like ice cream (85%, p1_1|ABC) and marshmallow treats (80%, p2_1|ABC).

Note that the QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where xx is any combination of numbers and '_', and YYY are any three capital letters). 
There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


Example output (strict JSON, no markdown, no code block, no extra text):
{{
  "SIMILAR_PATTERN_1": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}},
  "SIMILAR_PATTERN_2": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}},
  ...
  "DIFFERENT_PATTERN_5": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}}
}}

QUERY: {user_query}
RESULTS: {tmp_res_st}

{format_instructions}

Checklist before submitting:
- [ ] All fields are filled for each pattern.
- [ ] Each DESCRIPTION includes at least one number and QUESTION_ID.
- [ ] No field is left empty.
"""
    return prompt
prompt = create_prompt_crosssum(user_query="What does it mean to be Mexican?", tmp_res_st=tmp_res_st, n_topics=5, format_instructions=pattern_simdif_format_instructions)

print(prompt)


You are a research assistant analyzing survey results to answer the QUERY below.

Your job is to:
- Identify the top 5 SIMILAR PATTERNS (trends or agreements) and 5 DIFFERENT PATTERNS (contrasts or contradictions) relevant to the QUERY.
- For each pattern, provide:
    1. TITLE_SUMMARY: A short, descriptive title (never empty).
    2. VARIABLE_STRING: Comma-separated QUESTION_IDs used in the pattern (never empty).
    3. DESCRIPTION: A detailed explanation, citing numbers and QUESTION_IDs in parentheses (never empty).

Instructions:
- Do NOT leave any field empty. If information is limited, generalize or summarize what is available.
- Use only facts you are sure about, and always cite the QUESTION_ID for each fact.
- Do NOT repeat the same pattern in multiple fields.
- Ignore any results marked as 'NaN' or not available.
- If you combine categories (e.g., "like a lot" + "like somewhat"), only mention the sum if all original values are present.
- Do NOT invent data; if a pattern is wea

In [21]:
min(len(tmp_grade_dict) // 4, 5)

5

In [ ]:
import tiktoken
import re
import json

def get_answer(prompt, system_prompt=None, model='gpt-4o-mini-2024-07-18', temperature=0.9):
    # This function sends the prompt to the LLM and retrieves the response.
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user",   "content": prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


def clean_llm_json_output(content):
    # Remove code block markers and any leading/trailing text
    content = content.strip()
    # Remove code block markers
    content = re.sub(r"^```json", "", content, flags=re.IGNORECASE).strip()
    content = re.sub(r"^```", "", content, flags=re.IGNORECASE).strip()
    content = re.sub(r"```$", "", content, flags=re.IGNORECASE).strip()
    # Remove any leading text before the first {
    idx = content.find('{')
    if idx > 0:
        content = content[idx:]
    # Remove trailing text after last }
    idx2 = content.rfind('}')
    if idx2 > 0:
        content = content[:idx2+1]
    return content

def get_structured_summary(model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    # Pedir n patrones: tres resultados mínimos para cada patrón -- para minimimzar alucinaciones. 
    n_topics = min(len(tmp_grade_dict) // 4, 5)

    prompt =create_prompt_crosssum(user_query=user_query, tmp_res_st=tmp_res_st, n_topics=n_topics, format_instructions=pattern_simdif_format_instructions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_simdif.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_simdif.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}

def create_prompt_expt_smry(tst_lgc_dict, tmp_ky, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and survey summaries.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """

    ky = tmp_ky

    # Variables identified by the model
    tst_str_lst = tst_lgc_dict[ky]['VARIABLE_STRING'].split(',')
    tst_str_lst = [st + '__question' for st in tst_str_lst]
    print(f'Variables identified by the model: {tst_str_lst}')

    # Variables in the database
    tmp_db_var_lst = db_f1.get(ids=tst_str_lst)['ids']
    tmp_db_var_lst = list(set(tmp_db_var_lst))
    print(f'Variables in the database: {tmp_db_var_lst}')

    # Hallucinated variables
    tmp_hlc_var_lst = set(tst_str_lst) - set(tmp_db_var_lst)
    if tmp_hlc_var_lst:
        print(f'Hallucinated variables by the model: {tmp_hlc_var_lst}')

    # Implications for the identified variables
    tmp_imp_lst = [st.replace('question', 'implications') for st in tmp_db_var_lst]
    tmp_ky_lst = [st.split('__')[0] for st in tmp_imp_lst]
    tmp_imp_dict = dict(
        zip(
            tmp_ky_lst,
            db_f1.get(ids=tmp_imp_lst)['documents']
        )
    )

    imp_st = ' * '.join(tmp_imp_dict.values())
    tmp_smry = tst_lgc_dict[ky]['DESCRIPTION']

    # Generate the prompt
    prompt = f"""
    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do your work in either language. But you will reply in English.
    
    Your task is to read the EXPERT STATEMENTS from one or more experts about what information they consider important to receive from the survey results. 
    Then you will read a SURVEY SUMMARY of some survey results and will write a one-paragraph reply to the experts, elaborating on how the results relate and illustrate their concerns. 
    Note that they are experts and expect a detailed and thorough reply. Reply to all of them in the same paragraph, but be sure to address all of their points. 
    Do not refer to the experts or to yourself in the first person, just write the paragraph as if it was a report.

    Note that the SURVEY SUMMARY will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    They are not relevant to your answer, but you will need to keep track of these QUESTION_IDS of format pxx|YYY in your RETURN OBJECT.
    There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    EXPERT STATEMENTS: * {imp_st}
    SURVEY SUMMARY: {tmp_smry}

    {format_instructions}
    """
    return prompt

def get_structured_expert_summary(tst_lgc_dict, tmp_ky, model_name="gpt-4o-mini-2024-07-18", temperature=0.9):
    """
    Generates a structured expert summary for a given key in the logical group dictionary.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        model_name (str): Name of the LLM model to use.
        temperature (float): Temperature setting for the LLM.

    Returns:
        dict: Parsed response as a dictionary.
    """
    # Generate the prompt
    prompt = create_prompt_expt_smry(
        tst_lgc_dict=tst_lgc_dict,
        tmp_ky=tmp_ky,
        format_instructions=expert_summary_format_instructions
    )
    
    # Get the response from the model
    response = get_answer(prompt=prompt, model=model_name, temperature=temperature)
    
    # Parse the response
    parsed = expert_summary_parser.parse(response)
    
    # Ensure the parsed response is returned as a dictionary
    return parsed.model_dump()

def batch_process_expert_summaries(tst_lgc_dict, batch_size=8192):
    """
    Processes expert summaries in batches, saving checkpoints after each batch.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        batch_size (int): Maximum token limit for each batch.

    Returns:
        dict: A dictionary of structured results.
    """
    # Prepare prompts and keys
    prompts = [
        create_prompt_expt_smry(tst_lgc_dict, key, format_instructions=expert_summary_format_instructions)
        for key in tst_lgc_dict.keys()
    ]
    keys = list(tst_lgc_dict.keys())
    
    # Batch the documents
    batches = batch_documents(prompts, keys, max_tokens=batch_size, encoding_name="cl100k_base")
    
    # Initialize results dictionary
    structured_results = {}
    
    # Process each batch
    with tqdm.tqdm(total=len(keys), desc="Processing Expert Summaries") as pbar:
        for batch_docs, batch_keys in batches:
            for prompt, key in zip(batch_docs, batch_keys):
                try:
                    # Call get_structured_expert_summary for each key
                    structured_results[key] = get_structured_expert_summary(
                        tst_lgc_dict=tst_lgc_dict,
                        tmp_ky=key,
                        model_name="gpt-4o-mini-2024-07-18",
                        temperature=0.9
                    )
                except Exception as e:
                    structured_results[key] = {'error': str(e)}
                pbar.update(1)

    
    return structured_results

def create_prompt_trnsvl(tmp_smry_st, user_query, n_cmn_tpc, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and answering a query.

    Parameters:
        tmp_smry_st (str): The list of expert statements.
        user_query (str): The query to be answered.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """
    prompt = f"""
    You are a very thorough research assistant and expert in survey research and public opinion.
    You are fully bilingual in English and Spanish, and can do your work in either language.

    Your task is to perform three analyses and return a single Python dictionary with the results.

    1) Read the following list of STATEMENTS made by experts in several topics, which mention results, their implications, and their relevance.
    Each statement starts with the marker `*` and contains all information for a single statement: results, implications, and relevance.
    Identify {n_cmn_tpc} common topics across the STATEMENTS and write a one-paragraph summary of each topic, citing the most relevant numbers and explanations for each.
    Format your answer as a Python dictionary with the names of the topics as keys in ALL CAPS in the same language as the query, and the summaries as values.

    Note that the STATEMENTS will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.
    They are not relevant to your answer and your will use them only to identify the variables in the statements that your will use to write the summaries.

    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    Here are the statements: {tmp_smry_st}

    Save your summaries in a Python dictionary with the key `TOPIC_SUMMARIES`.

    2) Read your `TOPIC_SUMMARIES` and write a one-paragraph summary of the most relevant results and implications of the survey results, written for a general audience.
    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).
    Save your summary in a Python dictionary with the key `TOPIC_SUMMARY`.

    3) Read the QUERY and your `TOPIC_SUMMARY` and write a two-sentence answer to the QUERY that summarizes the most important points of your `TOPIC_SUMMARY`.
    Do not include any numbers or facts in your answer, just your reply to the QUERY.
    Be concise and do not repeat numbers; just answer the QUERY with the relevant ideas.
    
    Here is the QUERY: {user_query}

    Save your answer in a Python dictionary with the key `QUERY_ANSWER`.

    IMPORTANT: Your replies for all three tasks must be in the language of the QUERY.
    IMPORTANT: Make sure to return only a correctly formatted Python dictionary, without any code block formatting, markdown, or additional text.

    {format_instructions}
    """
    return prompt

def get_transversal_analysis(tmp_smry_st, user_query, model_name='gpt-4o-mini-2024-07-18', temperature=0.9):
    # Generate the prompt
    prompt = create_prompt_trnsvl(
        tmp_smry_st=tmp_smry_st,
        user_query=user_query,
        n_cmn_tpc=3,
        format_instructions=transversal_format_instructions
    )
    
    # Get the response from the model
    response = get_answer(prompt=prompt, model=model_name, temperature=temperature)
    parsed = transversal_parser.parse(response)
    # Parse the response
    return parsed.model_dump()  # Returns the parsed response as a dictionary.

# Parsers

pattern_parser_simdif = PydanticOutputParser(pydantic_object=FlatPatternSummary)
pattern_simdif_format_instructions = pattern_parser_simdif.get_format_instructions()

expert_summary_parser = PydanticOutputParser(pydantic_object=ExpertSummaryResponse)
expert_summary_format_instructions = expert_summary_parser.get_format_instructions()

transversal_parser = PydanticOutputParser(pydantic_object=TransversalAnalysisResponse)
transversal_format_instructions = transversal_parser.get_format_instructions()



def _deep_analyzer(tmp_pre_res_dict, tmp_grade_dict):
    """
    internal function to produce the transversal analysis and expert summaries
    Args: tmp_pre_res_dict (dict): Dictionary containing preprocessed results.
    Returns:
        tuple: A tuple containing two dictionaries:
        - tmp_preproc_dic: Filtered dictionary with relevant questions and summaries.
        - final_smry_dict: Dictionary with the final transversal analysis.
        - structured_expert_results: Dictionary with structured expert summaries.
    """

    tmp_preproc_dic = {k: v for k, v in tmp_pre_res_dict.items() if k.split('__')[1] in ['question', 'summary'] }

    # filtro de las preguntas relevantes
    tmp_preproc_dic ={k: v for k, v in tmp_preproc_dic.items() if any(k.startswith(grade_key) for grade_key in tmp_grade_dict.keys())}

    tmp_combined_strings = []

    for i, (k, v) in enumerate(tmp_preproc_dic.items(), start=1):
        facet = k.split("__", 1)[1].upper()
        p_id = k#.split("__")[0].split("|")[0]

        grouped_index = (i + 1) // 2 
        # q_id = v.split("|")[0]
        parts = v.split("|", 1)
        text = parts[1] if len(parts) > 1 else parts[0]

        p_id = p_id.split("__")[0]

        tmp_combined_strings.append(f"{facet}_{grouped_index}|{p_id}: {text}")

    tmp_res_st = '\n'.join(tmp_combined_strings)

    tst_lgc_dict = get_structured_summary(user_query=user_query, tmp_res_st=tmp_res_st, model_name = mod_alto, temperature= 0.0)

    structured_expert_results = batch_process_expert_summaries(tst_lgc_dict)

    tmp_smry_st = ' * '.join([v['EXPERT_REPLY'] for v in structured_expert_results.values()])

    final_smry_dict = get_transversal_analysis(tmp_smry_st, user_query)
    return tmp_preproc_dic, final_smry_dict, structured_expert_results

{
  "SIMILAR_PATTERN_1": {
    "TITLE_SUMMARY": "Strong National Pride Among Mexicans",
    "VARIABLE_STRING": "p6|IDE,p5|CUL,p15|GLO,p19_3|FED",
    "DESCRIPTION": "Across multiple questions, a majority of respondents express strong pride in being Mexican: 63.42% feel 'mucho' orgullo (p6|IDE), 59.67% feel 'mucho' orgullo (p5|CUL), 55.67% are 'very proud' and 33.58% 'proud' (totaling 89.25%, p15|GLO), and 63.58% are 'very proud' (p19_3|FED). This consistent high level of pride highlights a core element of Mexican identity."
  },
  "SIMILAR_PATTERN_2": {
    "TITLE_SUMMARY": "Pride as a Central Emotional Association with Mexico",
    "VARIABLE_STRING": "p1_1a_1|IDE,p5_1|IDE",
    "DESCRIPTION": "The most prominent association with 'México' is 'Orgullo' (pride) at 25.25% (p1_1a_1|IDE), and the predominant emotion felt about Mexico is also 'Pride' at 38.33% (p5_1|IDE). This demonstrates that pride is a central emotional and conceptual element in how Mexicans define their national identity

### generación de resúmenes expertos (structured_expert_results)

In [23]:
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

# Define the Pydantic model for the output
class ExpertSummaryResponse(BaseModel):
    EXPERT_REPLY: str

# Create the parser
expert_summary_parser = PydanticOutputParser(pydantic_object=ExpertSummaryResponse)

# Generate format instructions
expert_summary_format_instructions = expert_summary_parser.get_format_instructions()

In [24]:
tst_lgc_dict

{'SIMILAR_PATTERN_1': {'TITLE_SUMMARY': 'Strong National Pride Among Mexicans',
  'VARIABLE_STRING': 'p6|IDE,p5|CUL,p15|GLO,p19_3|FED',
  'DESCRIPTION': "Across multiple questions, a majority of respondents express strong pride in being Mexican: 63.42% feel 'mucho' orgullo (p6|IDE), 59.67% feel 'mucho' orgullo (p5|CUL), 55.67% are 'very proud' and 33.58% 'proud' (totaling 89.25%, p15|GLO), and 63.58% are 'very proud' (p19_3|FED). This consistent high level of pride highlights a core element of Mexican identity."},
 'SIMILAR_PATTERN_2': {'TITLE_SUMMARY': 'Pride as a Central Emotional Association with Mexico',
  'VARIABLE_STRING': 'p1_1a_1|IDE,p5_1|IDE',
  'DESCRIPTION': "The most prominent association with 'México' is 'Orgullo' (pride) at 25.25% (p1_1a_1|IDE), and the predominant emotion felt about Mexico is also 'Pride' at 38.33% (p5_1|IDE). This demonstrates that pride is a central emotional and conceptual element in how Mexicans define their national identity."},
 'SIMILAR_PATTERN_3'

In [25]:
def create_prompt_expt_smry(tst_lgc_dict, tmp_ky, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and survey summaries.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """

    ky = tmp_ky

    # Variables identified by the model
    tst_str_lst = tst_lgc_dict[ky]['VARIABLE_STRING'].split(',')
    tst_str_lst = [st + '__question' for st in tst_str_lst]
    print(f'Variables identified by the model: {tst_str_lst}')

    # Variables in the database
    tmp_db_var_lst = db_f1.get(ids=tst_str_lst)['ids']
    tmp_db_var_lst = list(set(tmp_db_var_lst))
    print(f'Variables in the database: {tmp_db_var_lst}')

    # Hallucinated variables
    tmp_hlc_var_lst = set(tst_str_lst) - set(tmp_db_var_lst)
    if tmp_hlc_var_lst:
        print(f'Hallucinated variables by the model: {tmp_hlc_var_lst}')

    # Implications for the identified variables
    tmp_imp_lst = [st.replace('question', 'implications') for st in tmp_db_var_lst]
    tmp_ky_lst = [st.split('__')[0] for st in tmp_imp_lst]
    tmp_imp_dict = dict(
        zip(
            tmp_ky_lst,
            db_f1.get(ids=tmp_imp_lst)['documents']
        )
    )

    imp_st = ' * '.join(tmp_imp_dict.values())
    tmp_smry = tst_lgc_dict[ky]['DESCRIPTION']

    # Generate the prompt
    prompt = f"""
    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do your work in either language. But you will reply in English.
    
    Your task is to read the EXPERT STATEMENTS from one or more experts about what information they consider important to receive from the survey results. 
    Then you will read a SURVEY SUMMARY of some survey results and will write a one-paragraph reply to the experts, elaborating on how the results relate and illustrate their concerns. 
    Note that they are experts and expect a detailed and thorough reply. Reply to all of them in the same paragraph, but be sure to address all of their points. 
    Do not refer to the experts or to yourself in the first person, just write the paragraph as if it was a report.

    Note that the SURVEY SUMMARY will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    They are not relevant to your answer, but you will need to keep track of these QUESTION_IDS of format pxx|YYY in your RETURN OBJECT.
    There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    EXPERT STATEMENTS: * {imp_st}
    SURVEY SUMMARY: {tmp_smry}

    {format_instructions}
    """
    return prompt

tmp_ky = 'SIMILAR_PATTERN_1'

# Generate the prompt
prompt = create_prompt_expt_smry(
    tst_lgc_dict=tst_lgc_dict,
    tmp_ky=tmp_ky,
    format_instructions=expert_summary_format_instructions
)

print(prompt)

Variables identified by the model: ['p6|IDE__question', 'p5|CUL__question', 'p15|GLO__question', 'p19_3|FED__question']
Variables in the database: ['p19_3|FED__question', 'p6|IDE__question', 'p15|GLO__question', 'p5|CUL__question']

    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do your work in either language. But you will reply in English.

    Your task is to read the EXPERT STATEMENTS from one or more experts about what information they consider important to receive from the survey results. 
    Then you will read a SURVEY SUMMARY of some survey results and will write a one-paragraph reply to the experts, elaborating on how the results relate and illustrate their concerns. 
    Note that they are experts and expect a detailed and thorough reply. Reply to all of them in the same paragraph, but b

## report_generator

In [ ]:
# tmp_res_st es el string con las variables filtradas

tmp_preproc_dic = {k: v for k, v in tmp_pre_res_dict.items() if k.split('__')[1] in ['question', 'summary'] }

# filtro de las preguntas relevantes
tmp_preproc_dic ={k: v for k, v in tmp_preproc_dic.items() if any(k.startswith(grade_key) for grade_key in tmp_grade_dict.keys())}

tmp_combined_strings = []

for i, (k, v) in enumerate(tmp_preproc_dic.items(), start=1):
    facet = k.split("__", 1)[1].upper()
    p_id = k#.split("__")[0].split("|")[0]

    grouped_index = (i + 1) // 2 
    # only split once, and if there is no "|" take the entire v
    q_id = v.split("|")[0]
    parts = v.split("|", 1)
    text = parts[1] if len(parts) > 1 else parts[0]

    p_id = p_id.split("__")[0]

    tmp_combined_strings.append(f"{facet}_{grouped_index}|{p_id}: {text}")

tmp_res_st = '\n'.join(tmp_combined_strings)
tmp_res_st

### investigación de patrones lógicos (tst_lgc_dict)

In [ ]:
## generación del esquema de pydantic para la respuesta

from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser
from typing import Dict

# 1. Define Pydantic model for an entry

class PatternItem(BaseModel):
    TITLE_SUMMARY: str
    VARIABLE_STRING: str
    DESCRIPTION: str

class FlatPatternSummary(BaseModel):
    """
    A “catch-all” model so we can declare SIMILAR_PATTERN_1…n
    and DIFFERENT_PATTERN_1…n at the top level.
    """
    class Config:
        extra = "allow"


def flat_pattern_template(n: int) -> FlatPatternSummary:
    """
    Build an “empty” flat summary with n SIMILAR_PATTERN_i
    and n DIFFERENT_PATTERN_i placeholders.
    """
    empty = {"TITLE_SUMMARY": "", "VARIABLE_STRING": "", "DESCRIPTION": ""}
    payload: dict[str, dict[str, str]] = {}
    for kind in ("SIMILAR_PATTERN", "DIFFERENT_PATTERN"):
        for i in range(1, n + 1):
            payload[f"{kind}_{i}"] = empty.copy()
    return FlatPatternSummary(**payload)


# example: build a 3×3 skeleton and print its JSON
tmpl = flat_pattern_template(3)
# print(tmpl.model_dump_json(indent=2))

from langchain.output_parsers import PydanticOutputParser

# re-create your parser against the PatternSummaryEntry template
pattern_parser_simdif = PydanticOutputParser(pydantic_object=FlatPatternSummary)
pattern_simdif_format_instructions = pattern_parser_simdif.get_format_instructions()

# optional: inspect the format instructions
print(pattern_simdif_format_instructions)



In [ ]:
def create_prompt_crosssum(user_query, tmp_res_st, n_topics=5, format_instructions=""):
    """
    Optimized prompt for extracting non-empty, detailed patterns from survey results.
    """
    prompt = f"""
You are a research assistant analyzing survey results to answer the QUERY below.

Your job is to:
- Identify the top {n_topics} SIMILAR PATTERNS (trends or agreements) and {n_topics} DIFFERENT PATTERNS (contrasts or contradictions) relevant to the QUERY.
- For each pattern, provide:
    1. TITLE_SUMMARY: A short, descriptive title (never empty).
    2. VARIABLE_STRING: Comma-separated QUESTION_IDs used in the pattern (never empty).
    3. DESCRIPTION: A detailed explanation, citing numbers and QUESTION_IDs in parentheses (never empty).

Instructions:
- Do NOT leave any field empty. If information is limited, generalize or summarize what is available.
- Use only facts you are sure about, and always cite the QUESTION_ID for each fact.
- Do NOT repeat the same pattern in multiple fields.
- Ignore any results marked as 'NaN' or not available.
- If you combine categories (e.g., "like a lot" + "like somewhat"), only mention the sum if all original values are present.
- Do NOT invent data; if a pattern is weak, explain that.
- Each pattern must be unique and relevant to the QUERY.

Example input format:
* QUESTION_ID: p1_1|ABC
  QUESTION: 'Do you like ice cream?'
  SUMMARY: 85% of people like ice cream, 10% do not, 5% do not know.
* QUESTION_ID: p2_1|ABC
  QUESTION: 'Do you like marshmallow treats?'
  SUMMARY: 80% like, 15% do not, 5% do not know.
* QUESTION_ID: p10_1|DEF:
  QUESTION: 'Do you like sour apple candies?'
  SUMMARY: 40% of people like sour apple candies, while 50% said they do not like it, and 10% said they do not know.

In these cases, the SIMILAR PATTERN is that 'people really like sweet treats because 85% like ice cream and 80% like marshmallow treats', 
and the DIFFERENT PATTERN is that 'people do not like sour treats as much because only 40% said they liked them'.

Example output for a SIMILAR PATTERN:
TITLE_SUMMARY: High preference for sweet treats
VARIABLE_STRING: p1_1|ABC,p2_1|ABC
DESCRIPTION: A large majority like ice cream (85%, p1_1|ABC) and marshmallow treats (80%, p2_1|ABC).

Note that the QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where xx is any combination of numbers and '_', and YYY are any three capital letters). 
There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


Example output (strict JSON, no markdown, no code block, no extra text):
{{
  "SIMILAR_PATTERN_1": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}},
  "SIMILAR_PATTERN_2": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}},
  ...
  "DIFFERENT_PATTERN_5": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}}
}}

QUERY: {user_query}
RESULTS: {tmp_res_st}

{format_instructions}

Checklist before submitting:
- [ ] All fields are filled for each pattern.
- [ ] Each DESCRIPTION includes at least one number and QUESTION_ID.
- [ ] No field is left empty.
"""
    return prompt
prompt = create_prompt_crosssum(user_query="What does it mean to be Mexican?", tmp_res_st=tmp_res_st, n_topics=5, format_instructions=pattern_simdif_format_instructions)

print(prompt)

In [ ]:
min(len(tmp_grade_dict) // 4, 5)

In [ ]:
import tiktoken
import re
import json

def get_answer(prompt, system_prompt=None, model='gpt-4o-mini-2024-07-18', temperature=0.9):
    # This function sends the prompt to the LLM and retrieves the response.
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user",   "content": prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


def clean_llm_json_output(content):
    # Remove code block markers and any leading/trailing text
    content = content.strip()
    # Remove code block markers
    content = re.sub(r"^```json", "", content, flags=re.IGNORECASE).strip()
    content = re.sub(r"^```", "", content, flags=re.IGNORECASE).strip()
    content = re.sub(r"```$", "", content, flags=re.IGNORECASE).strip()
    # Remove any leading text before the first {
    idx = content.find('{')
    if idx > 0:
        content = content[idx:]
    # Remove trailing text after last }
    idx2 = content.rfind('}')
    if idx2 > 0:
        content = content[:idx2+1]
    return content

def get_structured_summary(model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    # Pedir n patrones: tres resultados mínimos para cada patrón -- para minimimzar alucinaciones. 
    n_topics = min(len(tmp_grade_dict) // 4, 5)

    prompt =create_prompt_crosssum(user_query=user_query, tmp_res_st=tmp_res_st, n_topics=n_topics, format_instructions=pattern_simdif_format_instructions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_simdif.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_simdif.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}

def create_prompt_expt_smry(tst_lgc_dict, tmp_ky, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and survey summaries.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """

    ky = tmp_ky

    # Variables identified by the model
    tst_str_lst = tst_lgc_dict[ky]['VARIABLE_STRING'].split(',')
    tst_str_lst = [st + '__question' for st in tst_str_lst]
    print(f'Variables identified by the model: {tst_str_lst}')

    # Variables in the database
    tmp_db_var_lst = db_f1.get(ids=tst_str_lst)['ids']
    tmp_db_var_lst = list(set(tmp_db_var_lst))
    print(f'Variables in the database: {tmp_db_var_lst}')

    # Hallucinated variables
    tmp_hlc_var_lst = set(tst_str_lst) - set(tmp_db_var_lst)
    if tmp_hlc_var_lst:
        print(f'Hallucinated variables by the model: {tmp_hlc_var_lst}')

    # Implications for the identified variables
    tmp_imp_lst = [st.replace('question', 'implications') for st in tmp_db_var_lst]
    tmp_ky_lst = [st.split('__')[0] for st in tmp_imp_lst]
    tmp_imp_dict = dict(
        zip(
            tmp_ky_lst,
            db_f1.get(ids=tmp_imp_lst)['documents']
        )
    )

    imp_st = ' * '.join(tmp_imp_dict.values())
    tmp_smry = tst_lgc_dict[ky]['DESCRIPTION']

    # Generate the prompt
    prompt = f"""
    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do your work in either language. But you will reply in English.
    
    Your task is to read the EXPERT STATEMENTS from one or more experts about what information they consider important to receive from the survey results. 
    Then you will read a SURVEY SUMMARY of some survey results and will write a one-paragraph reply to the experts, elaborating on how the results relate and illustrate their concerns. 
    Note that they are experts and expect a detailed and thorough reply. Reply to all of them in the same paragraph, but be sure to address all of their points. 
    Do not refer to the experts or to yourself in the first person, just write the paragraph as if it was a report.

    Note that the SURVEY SUMMARY will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    They are not relevant to your answer, but you will need to keep track of these QUESTION_IDS of format pxx|YYY in your RETURN OBJECT.
    There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    EXPERT STATEMENTS: * {imp_st}
    SURVEY SUMMARY: {tmp_smry}

    {format_instructions}
    """
    return prompt

def get_structured_summary_grader(tmp_svvinfo_st, model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    prompt =create_prompt_grader(user_query=user_query, tmp_svvinfo_st= tmp_svvinfo_st, format_instructions=pattern_format_grader_instrtuctions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_grader.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_grader.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}



tmp_ky = 15
tmp_id_st = top_ids[tmp_ky]
tmp_svyinfo_dict = {k.split('__')[1].upper(): v for k,v in tmp_pre_res_dict.items() if k.startswith(tmp_id_st)}
tmp_svvinfo_st = ' '.join([f'{k}: {v}' for k,v in tmp_svyinfo_dict.items()])


content, tmp_grade_dict = get_structured_summary_grader(tmp_svvinfo_st, model_name = mod_bajo, temperature= 0.1)
print(content)

# TODO: batch queries, review grades, test new queries and models. 


In [ ]:
def create_tmp_svyinfo_dict(tmp_ky, top_ids, tmp_pre_res_dict):
    """         
    Create a temporary survey information dictionary for a given key.
    Args:
        tmp_ky (int): The index of the key in the top_ids list.
        top_ids (list): The list of top IDs.
        tmp_pre_res_dict (dict): The dictionary containing preprocessed results.
    Returns:

        dict: A dictionary containing the survey information for the specified key.
    """
    # Create a temporary survey information dictionary for a given key
    tmp_id_st = top_ids[tmp_ky]
    tmp_svyinfo_dict = {k.split('__')[1].upper(): v for k,v in tmp_pre_res_dict.items() if k.startswith(tmp_id_st)}
    tmp_svvinfo_st = ' '.join([f'{k}: {v}' for k,v in tmp_svyinfo_dict.items()])
    return tmp_svvinfo_st

# Example usage
tmp_ky = 1
tmp_svyinfo_dict = create_tmp_svyinfo_dict(tmp_ky, top_ids, tmp_pre_res_dict)
print(tmp_svyinfo_dict)

In [ ]:
def get_structured_summary_grader_p(prompt, model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    # prompt =create_prompt_grader(user_query=user_query, tmp_svvinfo_st= tmp_svvinfo_st, format_instructions=pattern_format_grader_instrtuctions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_grader.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_grader.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}
    
def create_tmp_svyinfo_dict(tmp_ky, top_ids, tmp_pre_res_dict):
    """         
    Create a temporary survey information dictionary for a given key.
    Args:
        tmp_ky (int): The index of the key in the top_ids list.
        top_ids (list): The list of top IDs.
        tmp_pre_res_dict (dict): The dictionary containing preprocessed results.
    Returns:

        dict: A dictionary containing the survey information for the specified key.
    """
    # Create a temporary survey information dictionary for a given key
    tmp_id_st = top_ids[tmp_ky]
    tmp_svyinfo_dict = {k.split('__')[1].upper(): v for k,v in tmp_pre_res_dict.items() if k.startswith(tmp_id_st)}
    tmp_svvinfo_st = ' '.join([f'{k}: {v}' for k,v in tmp_svyinfo_dict.items()])
    return tmp_svvinfo_st

def batch_process_expert_grader(user_query, top_ids, tmp_pre_res_dict, model_name , batch_size=8192):
    """
    Processes expert summaries in batches, saving checkpoints after each batch.

    Parameters:
        top_ids (list): List of variables to grade.
        tmp_pre_res_dict (dict): Dict with survey results.
        batch_size (int): Maximum token limit for each batch.

    Returns:
        dict: A dictionary of structured results.
    """


    # Prepare prompts and keys
    prompts = [
        create_prompt_grader(user_query,  
                             create_tmp_svyinfo_dict(tmp_ky, top_ids, 
                                                     tmp_pre_res_dict),
                                                     format_instructions=pattern_format_grader_instrtuctions)
        for tmp_ky in range(len(top_ids))
    ]
    keys = top_ids
    
    # Batch the documents
    batches = batch_documents(prompts, keys, max_tokens=batch_size, encoding_name="cl100k_base")
    
    # Initialize results dictionary
    structured_results = {}
    
    # Process each batch
    with tqdm.tqdm(total=len(keys), desc="Processing Expert Summaries") as pbar:
        for batch_docs, batch_keys in batches:
            for prompt, key in zip(batch_docs, batch_keys):
                try:
                    _, tmp_grade_dict = get_structured_summary_grader_p(prompt, model_name = model_name, temperature= 0.5)
                    #print(f'tmp_grade_dict: {tmp_grade_dict}')
                    structured_results[key] = tmp_grade_dict
                except Exception as e:
                    structured_results[key] = {'error': str(e)}
                pbar.update(1)
    
    return structured_results

tst_res = batch_process_expert_grader(user_query, top_ids, tmp_pre_res_dict, mod_alto, batch_size=8192)
tmp_grade_dict=  {k: v['GRADE_DICT'] for k, v in tst_res.items()}

# Filtrar los elementos que tienen una calificación mayor a 1

tmp_grade_dict= {k: v for k, v in tmp_grade_dict.items() if list(v.keys())[0] >1 }
tmp_grade_dict

## report_generator

In [ ]:
# tmp_res_st es el string con las variables filtradas

tmp_preproc_dic = {k: v for k, v in tmp_pre_res_dict.items() if k.split('__')[1] in ['question', 'summary'] }

# filtro de las preguntas relevantes
tmp_preproc_dic ={k: v for k, v in tmp_preproc_dic.items() if any(k.startswith(grade_key) for grade_key in tmp_grade_dict.keys())}

tmp_combined_strings = []

for i, (k, v) in enumerate(tmp_preproc_dic.items(), start=1):
    facet = k.split("__", 1)[1].upper()
    p_id = k#.split("__")[0].split("|")[0]

    grouped_index = (i + 1) // 2 
    # only split once, and if there is no "|" take the entire v
    q_id = v.split("|")[0]
    parts = v.split("|", 1)
    text = parts[1] if len(parts) > 1 else parts[0]

    p_id = p_id.split("__")[0]

    tmp_combined_strings.append(f"{facet}_{grouped_index}|{p_id}: {text}")

tmp_res_st = '\n'.join(tmp_combined_strings)
tmp_res_st

### investigación de patrones lógicos (tst_lgc_dict)

In [ ]:
## generación del esquema de pydantic para la respuesta

from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser
from typing import Dict

# 1. Define Pydantic model for an entry

class PatternItem(BaseModel):
    TITLE_SUMMARY: str
    VARIABLE_STRING: str
    DESCRIPTION: str

class FlatPatternSummary(BaseModel):
    """
    A “catch-all” model so we can declare SIMILAR_PATTERN_1…n
    and DIFFERENT_PATTERN_1…n at the top level.
    """
    class Config:
        extra = "allow"


def flat_pattern_template(n: int) -> FlatPatternSummary:
    """
    Build an “empty” flat summary with n SIMILAR_PATTERN_i
    and n DIFFERENT_PATTERN_i placeholders.
    """
    empty = {"TITLE_SUMMARY": "", "VARIABLE_STRING": "", "DESCRIPTION": ""}
    payload: dict[str, dict[str, str]] = {}
    for kind in ("SIMILAR_PATTERN", "DIFFERENT_PATTERN"):
        for i in range(1, n + 1):
            payload[f"{kind}_{i}"] = empty.copy()
    return FlatPatternSummary(**payload)


# example: build a 3×3 skeleton and print its JSON
tmpl = flat_pattern_template(3)
# print(tmpl.model_dump_json(indent=2))

from langchain.output_parsers import PydanticOutputParser

# re-create your parser against the PatternSummaryEntry template
pattern_parser_simdif = PydanticOutputParser(pydantic_object=FlatPatternSummary)
pattern_simdif_format_instructions = pattern_parser_simdif.get_format_instructions()

# optional: inspect the format instructions
print(pattern_simdif_format_instructions)



In [ ]:
def create_prompt_crosssum(user_query, tmp_res_st, n_topics=5, format_instructions=""):
    """
    Optimized prompt for extracting non-empty, detailed patterns from survey results.
    """
    prompt = f"""
You are a research assistant analyzing survey results to answer the QUERY below.

Your job is to:
- Identify the top {n_topics} SIMILAR PATTERNS (trends or agreements) and {n_topics} DIFFERENT PATTERNS (contrasts or contradictions) relevant to the QUERY.
- For each pattern, provide:
    1. TITLE_SUMMARY: A short, descriptive title (never empty).
    2. VARIABLE_STRING: Comma-separated QUESTION_IDs used in the pattern (never empty).
    3. DESCRIPTION: A detailed explanation, citing numbers and QUESTION_IDs in parentheses (never empty).

Instructions:
- Do NOT leave any field empty. If information is limited, generalize or summarize what is available.
- Use only facts you are sure about, and always cite the QUESTION_ID for each fact.
- Do NOT repeat the same pattern in multiple fields.
- Ignore any results marked as 'NaN' or not available.
- If you combine categories (e.g., "like a lot" + "like somewhat"), only mention the sum if all original values are present.
- Do NOT invent data; if a pattern is weak, explain that.
- Each pattern must be unique and relevant to the QUERY.

Example input format:
* QUESTION_ID: p1_1|ABC
  QUESTION: 'Do you like ice cream?'
  SUMMARY: 85% of people like ice cream, 10% do not, 5% do not know.
* QUESTION_ID: p2_1|ABC
  QUESTION: 'Do you like marshmallow treats?'
  SUMMARY: 80% like, 15% do not, 5% do not know.
* QUESTION_ID: p10_1|DEF:
  QUESTION: 'Do you like sour apple candies?'
  SUMMARY: 40% of people like sour apple candies, while 50% said they do not like it, and 10% said they do not know.

In these cases, the SIMILAR PATTERN is that 'people really like sweet treats because 85% like ice cream and 80% like marshmallow treats', 
and the DIFFERENT PATTERN is that 'people do not like sour treats as much because only 40% said they liked them'.

Example output for a SIMILAR PATTERN:
TITLE_SUMMARY: High preference for sweet treats
VARIABLE_STRING: p1_1|ABC,p2_1|ABC
DESCRIPTION: A large majority like ice cream (85%, p1_1|ABC) and marshmallow treats (80%, p2_1|ABC).

Note that the QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where xx is any combination of numbers and '_', and YYY are any three capital letters). 
There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


Example output (strict JSON, no markdown, no code block, no extra text):
{{
  "SIMILAR_PATTERN_1": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}},
  "SIMILAR_PATTERN_2": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}},
  ...
  "DIFFERENT_PATTERN_5": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}}
}}

QUERY: {user_query}
RESULTS: {tmp_res_st}

{format_instructions}

Checklist before submitting:
- [ ] All fields are filled for each pattern.
- [ ] Each DESCRIPTION includes at least one number and QUESTION_ID.
- [ ] No field is left empty.
"""
    return prompt
prompt = create_prompt_crosssum(user_query="What does it mean to be Mexican?", tmp_res_st=tmp_res_st, n_topics=5, format_instructions=pattern_simdif_format_instructions)

print(prompt)

In [ ]:
min(len(tmp_grade_dict) // 4, 5)

In [ ]:
import tiktoken
import re
import json

def get_answer(prompt, system_prompt=None, model='gpt-4o-mini-2024-07-18', temperature=0.9):
    # This function sends the prompt to the LLM and retrieves the response.
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user",   "content": prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


def clean_llm_json_output(content):
    # Remove code block markers and any leading/trailing text
    content = content.strip()
    # Remove code block markers
    content = re.sub(r"^```json", "", content, flags=re.IGNORECASE).strip()
    content = re.sub(r"^```", "", content, flags=re.IGNORECASE).strip()
    content = re.sub(r"```$", "", content, flags=re.IGNORECASE).strip()
    # Remove any leading text before the first {
    idx = content.find('{')
    if idx > 0:
        content = content[idx:]
    # Remove trailing text after last }
    idx2 = content.rfind('}')
    if idx2 > 0:
        content = content[:idx2+1]
    return content

def get_structured_summary(model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    # Pedir n patrones: tres resultados mínimos para cada patrón -- para minimimzar alucinaciones. 
    n_topics = min(len(tmp_grade_dict) // 4, 5)

    prompt =create_prompt_crosssum(user_query=user_query, tmp_res_st=tmp_res_st, n_topics=n_topics, format_instructions=pattern_simdif_format_instructions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_simdif.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_simdif.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}

def create_prompt_expt_smry(tst_lgc_dict, tmp_ky, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and survey summaries.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """

    ky = tmp_ky

    # Variables identified by the model
    tst_str_lst = tst_lgc_dict[ky]['VARIABLE_STRING'].split(',')
    tst_str_lst = [st + '__question' for st in tst_str_lst]
    print(f'Variables identified by the model: {tst_str_lst}')

    # Variables in the database
    tmp_db_var_lst = db_f1.get(ids=tst_str_lst)['ids']
    tmp_db_var_lst = list(set(tmp_db_var_lst))
    print(f'Variables in the database: {tmp_db_var_lst}')

    # Hallucinated variables
    tmp_hlc_var_lst = set(tst_str_lst) - set(tmp_db_var_lst)
    if tmp_hlc_var_lst:
        print(f'Hallucinated variables by the model: {tmp_hlc_var_lst}')

    # Implications for the identified variables
    tmp_imp_lst = [st.replace('question', 'implications') for st in tmp_db_var_lst]
    tmp_ky_lst = [st.split('__')[0] for st in tmp_imp_lst]
    tmp_imp_dict = dict(
        zip(
            tmp_ky_lst,
            db_f1.get(ids=tmp_imp_lst)['documents']
        )
    )

    imp_st = ' * '.join(tmp_imp_dict.values())
    tmp_smry = tst_lgc_dict[ky]['DESCRIPTION']

    # Generate the prompt
    prompt = f"""
    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do your work in either language. But you will reply in English.
    
    Your task is to read the EXPERT STATEMENTS from one or more experts about what information they consider important to receive from the survey results. 
    Then you will read a SURVEY SUMMARY of some survey results and will write a one-paragraph reply to the experts, elaborating on how the results relate and illustrate their concerns. 
    Note that they are experts and expect a detailed and thorough reply. Reply to all of them in the same paragraph, but be sure to address all of their points. 
    Do not refer to the experts or to yourself in the first person, just write the paragraph as if it was a report.

    Note that the SURVEY SUMMARY will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    They are not relevant to your answer, but you will need to keep track of these QUESTION_IDS of format pxx|YYY in your RETURN OBJECT.
    There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    EXPERT STATEMENTS: * {imp_st}
    SURVEY SUMMARY: {tmp_smry}

    {format_instructions}
    """
    return prompt

def get_structured_summary_grader(tmp_svvinfo_st, model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    prompt =create_prompt_grader(user_query=user_query, tmp_svvinfo_st= tmp_svvinfo_st, format_instructions=pattern_format_grader_instrtuctions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_grader.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_grader.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}



tmp_ky = 15
tmp_id_st = top_ids[tmp_ky]
tmp_svyinfo_dict = {k.split('__')[1].upper(): v for k,v in tmp_pre_res_dict.items() if k.startswith(tmp_id_st)}
tmp_svvinfo_st = ' '.join([f'{k}: {v}' for k,v in tmp_svyinfo_dict.items()])


content, tmp_grade_dict = get_structured_summary_grader(tmp_svvinfo_st, model_name = mod_bajo, temperature= 0.1)
print(content)

# TODO: batch queries, review grades, test new queries and models. 


In [ ]:
def create_tmp_svyinfo_dict(tmp_ky, top_ids, tmp_pre_res_dict):
    """         
    Create a temporary survey information dictionary for a given key.
    Args:
        tmp_ky (int): The index of the key in the top_ids list.
        top_ids (list): The list of top IDs.
        tmp_pre_res_dict (dict): The dictionary containing preprocessed results.
    Returns:

        dict: A dictionary containing the survey information for the specified key.
    """
    # Create a temporary survey information dictionary for a given key
    tmp_id_st = top_ids[tmp_ky]
    tmp_svyinfo_dict = {k.split('__')[1].upper(): v for k,v in tmp_pre_res_dict.items() if k.startswith(tmp_id_st)}
    tmp_svvinfo_st = ' '.join([f'{k}: {v}' for k,v in tmp_svyinfo_dict.items()])
    return tmp_svvinfo_st

# Example usage
tmp_ky = 1
tmp_svyinfo_dict = create_tmp_svyinfo_dict(tmp_ky, top_ids, tmp_pre_res_dict)
print(tmp_svyinfo_dict)

In [ ]:
def get_structured_summary_grader_p(prompt, model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    # prompt =create_prompt_grader(user_query=user_query, tmp_svvinfo_st= tmp_svvinfo_st, format_instructions=pattern_format_grader_instrtuctions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_grader.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_grader.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}
    
def create_tmp_svyinfo_dict(tmp_ky, top_ids, tmp_pre_res_dict):
    """         
    Create a temporary survey information dictionary for a given key.
    Args:
        tmp_ky (int): The index of the key in the top_ids list.
        top_ids (list): The list of top IDs.
        tmp_pre_res_dict (dict): The dictionary containing preprocessed results.
    Returns:

        dict: A dictionary containing the survey information for the specified key.
    """
    # Create a temporary survey information dictionary for a given key
    tmp_id_st = top_ids[tmp_ky]
    tmp_svyinfo_dict = {k.split('__')[1].upper(): v for k,v in tmp_pre_res_dict.items() if k.startswith(tmp_id_st)}
    tmp_svvinfo_st = ' '.join([f'{k}: {v}' for k,v in tmp_svyinfo_dict.items()])
    return tmp_svvinfo_st

def batch_process_expert_grader(user_query, top_ids, tmp_pre_res_dict, model_name , batch_size=8192):
    """
    Processes expert summaries in batches, saving checkpoints after each batch.

    Parameters:
        top_ids (list): List of variables to grade.
        tmp_pre_res_dict (dict): Dict with survey results.
        batch_size (int): Maximum token limit for each batch.

    Returns:
        dict: A dictionary of structured results.
    """


    # Prepare prompts and keys
    prompts = [
        create_prompt_grader(user_query,  
                             create_tmp_svyinfo_dict(tmp_ky, top_ids, 
                                                     tmp_pre_res_dict),
                                                     format_instructions=pattern_format_grader_instrtuctions)
        for tmp_ky in range(len(top_ids))
    ]
    keys = top_ids
    
    # Batch the documents
    batches = batch_documents(prompts, keys, max_tokens=batch_size, encoding_name="cl100k_base")
    
    # Initialize results dictionary
    structured_results = {}
    
    # Process each batch
    with tqdm.tqdm(total=len(keys), desc="Processing Expert Summaries") as pbar:
        for batch_docs, batch_keys in batches:
            for prompt, key in zip(batch_docs, batch_keys):
                try:
                    _, tmp_grade_dict = get_structured_summary_grader_p(prompt, model_name = model_name, temperature= 0.5)
                    #print(f'tmp_grade_dict: {tmp_grade_dict}')
                    structured_results[key] = tmp_grade_dict
                except Exception as e:
                    structured_results[key] = {'error': str(e)}
                pbar.update(1)
    
    return structured_results

tst_res = batch_process_expert_grader(user_query, top_ids, tmp_pre_res_dict, mod_alto, batch_size=8192)
tmp_grade_dict=  {k: v['GRADE_DICT'] for k, v in tst_res.items()}

# Filtrar los elementos que tienen una calificación mayor a 1

tmp_grade_dict= {k: v for k, v in tmp_grade_dict.items() if list(v.keys())[0] >1 }
tmp_grade_dict

## report_generator

In [26]:
import tqdm
import tiktoken

def get_structured_expert_summary(tst_lgc_dict, tmp_ky, model_name="gpt-4o-mini-2024-07-18", temperature=0.9):
    """
    Generates a structured expert summary for a given key in the logical group dictionary.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        model_name (str): Name of the LLM model to use.
        temperature (float): Temperature setting for the LLM.

    Returns:
        dict: Parsed response as a dictionary.
    """
    # Generate the prompt
    prompt = create_prompt_expt_smry(
        tst_lgc_dict=tst_lgc_dict,
        tmp_ky=tmp_ky,
        format_instructions=expert_summary_format_instructions
    )
    
    # Get the response from the model
    response = get_answer(prompt=prompt, model=model_name, temperature=temperature)
    
    # Parse the response
    parsed = expert_summary_parser.parse(response)
    
    # Ensure the parsed response is returned as a dictionary
    return parsed.model_dump()


def num_tokens_from_string(string: str, encoding_name: str) -> int:
    # Utility function to calculate the number of tokens in a string.
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Update the token count calculation in batch_documents to use num_tokens_from_string
# 8192 is the token limit for the model
def batch_documents(docs, ids, max_tokens=8192, encoding_name="cl100k_base"):
    # This function batches documents and their IDs while respecting the token limit.
    # It ensures that each batch does not exceed the maximum token limit,
    # which is crucial for efficient processing with the LLM.
    batches = []
    current_batch_docs = []
    current_batch_ids = []
    current_token_count = 0

    for doc, doc_id in zip(docs, ids):
        doc_token_count = num_tokens_from_string(doc, encoding_name)  # Use the token counting function
        
        # print(f"Document ID: {doc_id}, Token Count: {doc_token_count}")
        
        if current_token_count + doc_token_count > max_tokens:
            # Save the current batch and start a new one
            batches.append((current_batch_docs, current_batch_ids))
            current_batch_docs = []
            current_batch_ids = []
            current_token_count = 0

        # Add the document to the current batch
        current_batch_docs.append(doc)
        current_batch_ids.append(doc_id)
        current_token_count += doc_token_count

    # Add the last batch if it has any documents
    if current_batch_docs:
        batches.append((current_batch_docs, current_batch_ids))

    return batches

def batch_process_expert_summaries(tst_lgc_dict, batch_size=8192):
    """
    Processes expert summaries in batches, saving checkpoints after each batch.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        batch_size (int): Maximum token limit for each batch.

    Returns:
        dict: A dictionary of structured results.
    """
    # Prepare prompts and keys
    prompts = [
        create_prompt_expt_smry(tst_lgc_dict, key, format_instructions=expert_summary_format_instructions)
        for key in tst_lgc_dict.keys()
    ]
    keys = list(tst_lgc_dict.keys())
    
    # Batch the documents
    batches = batch_documents(prompts, keys, max_tokens=batch_size, encoding_name="cl100k_base")
    
    # Initialize results dictionary
    structured_results = {}
    
    # Process each batch
    with tqdm.tqdm(total=len(keys), desc="Processing Expert Summaries") as pbar:
        for batch_docs, batch_keys in batches:
            for prompt, key in zip(batch_docs, batch_keys):
                try:
                    # Call get_structured_expert_summary for each key
                    structured_results[key] = get_structured_expert_summary(
                        tst_lgc_dict=tst_lgc_dict,
                        tmp_ky=key,
                        model_name="gpt-4o-mini-2024-07-18",
                        temperature=0.9
                    )
                except Exception as e:
                    structured_results[key] = {'error': str(e)}
                pbar.update(1)

    
    return structured_results

In [27]:

# Process expert summaries in batches
structured_expert_results = batch_process_expert_summaries(tst_lgc_dict)


Variables identified by the model: ['p6|IDE__question', 'p5|CUL__question', 'p15|GLO__question', 'p19_3|FED__question']
Variables in the database: ['p19_3|FED__question', 'p6|IDE__question', 'p15|GLO__question', 'p5|CUL__question']
Variables identified by the model: ['p1_1a_1|IDE__question', 'p5_1|IDE__question']
Variables in the database: ['p1_1a_1|IDE__question', 'p5_1|IDE__question']
Variables identified by the model: ['p1_1a_1|IDE__question', 'p13|IND__question', 'p13a_1|IND__question']
Variables in the database: ['p13a_1|IND__question', 'p1_1a_1|IDE__question', 'p13|IND__question']
Variables identified by the model: ['p7|IDE__question', 'p53|DEP__question']
Variables in the database: ['p53|DEP__question', 'p7|IDE__question']
Variables identified by the model: ['p2_1a_1|IDE__question']
Variables in the database: ['p2_1a_1|IDE__question']
Variables identified by the model: ['p6|IDE__question', 'p5|CUL__question', 'p15|GLO__question', 'p16_1|GLO__question', 'p16_2|GLO__question', 'p1

Processing Expert Summaries:   0%|          | 0/10 [00:00<?, ?it/s]

Variables identified by the model: ['p6|IDE__question', 'p5|CUL__question', 'p15|GLO__question', 'p19_3|FED__question']
Variables in the database: ['p19_3|FED__question', 'p6|IDE__question', 'p15|GLO__question', 'p5|CUL__question']


Processing Expert Summaries:  10%|█         | 1/10 [00:05<00:51,  5.70s/it]

Variables identified by the model: ['p1_1a_1|IDE__question', 'p5_1|IDE__question']
Variables in the database: ['p1_1a_1|IDE__question', 'p5_1|IDE__question']


...

Processing Expert Summaries:  80%|████████  | 8/10 [00:28<00:06,  3.32s/it]

Variables identified by the model: ['p2_1a_1|IDE__question', 'p13|IND__question', 'p13a_1|IND__question', 'p14a_1|IND__question']
Variables in the database: ['p13a_1|IND__question', 'p2_1a_1|IDE__question', 'p14a_1|IND__question', 'p13|IND__question']


Processing Expert Summaries:  90%|█████████ | 9/10 [00:31<00:03,  3.30s/it]

Variables identified by the model: ['p78|IDE__question', 'p67_1|MIG__question']
Variables in the database: ['p78|IDE__question', 'p67_1|MIG__question']


Processing Expert Summaries: 100%|██████████| 10/10 [00:35<00:00,  3.51s/it]


### Generación de resúmenes transversales y respuesta a query (final_smry_dict)

In [28]:
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

# Define the Pydantic model for the output
class TransversalAnalysisResponse(BaseModel):
    TOPIC_SUMMARIES: dict[str, str]  # A dictionary with topic names as keys and summaries as values
    TOPIC_SUMMARY: str  # A one-paragraph summary for a general audience
    QUERY_ANSWER: str  # A two-sentence answer to the query

# Create the parser
transversal_parser = PydanticOutputParser(pydantic_object=TransversalAnalysisResponse)

# Generate format instructions
transversal_format_instructions = transversal_parser.get_format_instructions()

def create_prompt_trnsvl(tmp_smry_st, user_query, n_cmn_tpc, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and answering a query.

    Parameters:
        tmp_smry_st (str): The list of expert statements.
        user_query (str): The query to be answered.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """
    prompt = f"""
    You are a very thorough research assistant and expert in survey research and public opinion.
    You are fully bilingual in English and Spanish, and can do your work in either language.

    Your task is to perform three analyses and return a single Python dictionary with the results.

    1) Read the following list of STATEMENTS made by experts in several topics, which mention results, their implications, and their relevance.
    Each statement starts with the marker `*` and contains all information for a single statement: results, implications, and relevance.
    Identify {n_cmn_tpc} common topics across the STATEMENTS and write a one-paragraph summary of each topic, citing the most relevant numbers and explanations for each.
    Format your answer as a Python dictionary with the names of the topics as keys in ALL CAPS in the same language as the query, and the summaries as values.

    Note that the STATEMENTS will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.
    They are not relevant to your answer and your will use them only to identify the variables in the statements that your will use to write the summaries.

    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    Here are the statements: {tmp_smry_st}

    Save your summaries in a Python dictionary with the key `TOPIC_SUMMARIES`.

    2) Read your `TOPIC_SUMMARIES` and write a one-paragraph summary of the most relevant results and implications of the survey results, written for a general audience.
    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).
    Save your summary in a Python dictionary with the key `TOPIC_SUMMARY`.

    3) Read the QUERY and your `TOPIC_SUMMARY` and write a two-sentence answer to the QUERY that summarizes the most important points of your `TOPIC_SUMMARY`.
    Do not include any numbers or facts in your answer, just your reply to the QUERY.
    Be concise and do not repeat numbers; just answer the QUERY with the relevant ideas.
    
    Here is the QUERY: {user_query}

    Save your answer in a Python dictionary with the key `QUERY_ANSWER`.

    IMPORTANT: Your replies for all three tasks must be in the language of the QUERY.
    IMPORTANT: Make sure to return only a correctly formatted Python dictionary, without any code block formatting, markdown, or additional text.

    {format_instructions}
    """
    return prompt


#

tmp_smry_st = ' * '.join([v['EXPERT_REPLY'] for v in structured_expert_results.values()])


def get_transversal_analysis(tmp_smry_st, user_query, model_name='gpt-4o-mini-2024-07-18', temperature=0.9):
    # Generate the prompt
    prompt = create_prompt_trnsvl(
        tmp_smry_st=tmp_smry_st,
        user_query=user_query,
        n_cmn_tpc=3,
        format_instructions=transversal_format_instructions
    )
    
    # Get the response from the model
    response = get_answer(prompt=prompt, model=model_name, temperature=temperature)
    parsed = transversal_parser.parse(response)
    # Parse the response
    return parsed.model_dump()  # Returns the parsed response as a dictionary.

# Example usage
final_smry_dict = get_transversal_analysis(tmp_smry_st, user_query)
final_smry_dict


{'TOPIC_SUMMARIES': {'IDENTIDAD Y VALORES': "Los resultados de la encuesta indican una fuerte conexión emocional con la identidad nacional en México, donde un 63.42% de los encuestados reportan 'mucho' orgullo (p6|IDE) y un 89.25% se consideran 'muy orgullosos' o 'orgullosos' (p15|GLO). Este sentimiento de orgullo es central para entender la identidad cultural y puede informar políticas y programas que promuevan la cohesión social.",
  'CULTURA POLITICA': "El 96.92% de los encuestados se identifican exclusivamente como 'Sólo mexicano' (p7|IDE), lo que resalta una identidad nacional robusta. Sin embargo, también se observó una baja satisfacción relacionada con instituciones como la democracia, donde solo un 7.17% se siente 'muy orgulloso' (p16_1|GLO), sugiriendo que las políticas deben enfocarse en mejorar la percepción de estas áreas para fortalecer la identidad y unidad nacional.",
  'DISCRIMINACION Y DESIGUALDAD': "Los hallazgos muestran preocupaciones significativas sobre la desigua

### Construcción de documento

In [29]:
import re

# extracción de plots

tmp_txt_lst = [final_smry_dict['TOPIC_SUMMARY']] + [st for st in final_smry_dict['TOPIC_SUMMARIES'].values()]


# Update pattern to match expanded format including letters in ID
pattern = r'\(([pP]\d+[a-zA-Z]?(?:_\d+)?(?:\|[A-Z0-9]+))\)'
# Extract matches from each text in tmp_txt_lst
tmp_matches = []
for text in tmp_txt_lst:
    matches = re.findall(pattern, text)
    if matches:
        tmp_matches.extend(matches)

# Remove duplicates while preserving order
tmp_id_lst = list(dict.fromkeys(tmp_matches))
# variables en los resultados que sí existen entre las que fueron pasadas al modelo:

print(f"Found {len(tmp_id_lst)} unique question IDs: {tmp_id_lst}")
# # Check for differences between the two sets
# print(f'variables identificadas pero no pasadas al modelo: {set(tmp_id_lst) - set(tmp_SEL_lst)}')
# print(f'variables pasadas al modelo pero no identificadas: {set(tmp_SEL_lst) - set(tmp_id_lst)}')


Found 6 unique question IDs: ['p6|IDE', 'p15|GLO', 'p7|IDE', 'p16_1|GLO', 'p32|POB', 'p14a_1|IND']


All IDs were found in df_tables.


In [ ]:
# creación de gráficos

import numpy as np
import math
import matplotlib.pyplot as plt

def split_text_by_words(text, n=8, k=50):
    """
    Splits a string into lines with a maximum of n words per line.
    If the first n-1 words already exceed k characters, truncates with '...'.
    """
    words = text.split()
    # if first (n-1) words already too long → truncate
    first_chunk = words[: n-1]
    if len(" ".join(first_chunk)) >= k:
        return " ".join(first_chunk) + "..."
    # otherwise chunk every n words
    lines = []
    for i in range(0, len(words), n):
        chunk = words[i : i + n]
        lines.append(" ".join(chunk))
    return "\n".join(lines)

def create_plot(df, question, figsize=(6, 8)):
    """
    Creates a bar plot (horizontal if rows <= 6, vertical otherwise) of the survey data.

    Args:
        df (pd.DataFrame): DataFrame containing the data to plot.
        question (str): The survey question to use as the plot title.

    Returns:
        matplotlib.pyplot.figure: The matplotlib figure object.
    """
    # Determine plot type based on the number of rows
    if len(df) >= 6:
        # Horizontal bar plot
        fig, ax = plt.subplots(figsize=figsize)
        
        # Create a colormap where 'No sabe/ No contesta' is always gray

        colormaps = [plt.cm.summer, plt.cm.spring, plt.cm.winter, plt.cm.autumn]
        selected_colormap = np.random.choice(colormaps)
        colors = selected_colormap(np.linspace(0, 0.8, len(df)))

        if 'No sabe/ No contesta' in df.index:
            gray_index = list(df.index).index('No sabe/ No contesta')
            colors[gray_index] = [0.7, 0.7, 0.7, 1.0]  # RGBA for gray
        bars = ax.barh(df.index, df['%'], color=colors)

        # Add percentage labels to the bars
        for bar in bars:
            width = bar.get_width()
            label_position = width + 0.5  # Slightly to the right of the bar
            ax.text(label_position, bar.get_y() + bar.get_height()/2,
                    f'{width:.1f}%',
                    ha='left', va='center', fontsize=8)
        # Customize plot
        ax.set_xlabel('Porcentaje (%)', fontsize=8)
        ax.set_title(split_text_by_words(question), fontsize=9)
        # Apply split_text_by_words to y-tick labels with conditional fontsize
        y_labels = [split_text_by_words(label, n=8, k=50) for label in df.index]
        y_fontsizes = [10 if len(label.split()) <= 4 else 8 for label in df.index]
        ax.set_yticklabels(y_labels)
        for label, fontsize in zip(ax.get_yticklabels(), y_fontsizes):
            label.set_fontsize(fontsize)

        # Apply split_text_by_words to x-tick labels with conditional fontsize
        x_labels = [split_text_by_words(str(label), n=8, k=50) for label in ax.get_xticks()]
        x_fontsizes = [10 if len(str(label).split()) <= 4 else 8 for label in ax.get_xticks()]
        ax.set_xticklabels(x_labels)
        for label, fontsize in zip(ax.get_xticklabels(), x_fontsizes):
            label.set_fontsize(fontsize)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Invert the y-axis to show the first item at the top
        ax.invert_yaxis()
        plt.tight_layout()

    else:
        # Vertical bar plot
        fig, ax = plt.subplots(figsize=figsize)

        colormaps = [plt.cm.summer, plt.cm.spring, plt.cm.winter, plt.cm.autumn]
        selected_colormap = np.random.choice(colormaps)
        colors = selected_colormap(np.linspace(0, 0.8, len(df)))

        if 'No sabe/ No contesta' in df.index:
            gray_index = list(df.index).index('No sabe/ No contesta')
            colors[gray_index] = [0.5, 0.5, 0.5, 1.0]  # RGBA for gray

        bars = ax.bar(df.index, df['%'], color=colors)

        # Add percentage labels above the bars
        for bar in bars:
            height = bar.get_height()
            label_position = height + 0.5
            ax.text(bar.get_x() + bar.get_width()/2, label_position,
                    f'{height:.1f}%',
                    ha='center', va='bottom', fontsize=8)

        # Customize plot
        ax.set_ylabel('Porcentaje (%)', fontsize=8)

        # Apply split_text_by_words to y-tick labels with conditional fontsize
        y_labels = [split_text_by_words(label, n=5, k=40) for label in df.index]
        y_fontsizes = [10 if len(label.split()) <= 4 else 6 for label in df.index]
        ax.set_yticklabels(y_labels)
        for label, fontsize in zip(ax.get_yticklabels(), y_fontsizes):
            label.set_fontsize(fontsize)

        # Apply split_text_by_words to x-tick labels with conditional fontsize
        x_labels = [split_text_by_words(str(label), n=8, k=50) for label in ax.get_xticks()]
        x_fontsizes = [10 if len(str(label).split()) <= 4 else 8 for label in ax.get_xticks()]
        ax.set_xticklabels(x_labels)
        for label, fontsize in zip(ax.get_xticklabels(), x_fontsizes):
            label.set_fontsize(fontsize)
        ax.set_title(split_text_by_words(question, 8), fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()

    return fig

def create_multiplot(dfs, questions=None, plot_idx=1, figsize=(10,5)):
    """
    Creates a grid of subplots (2 cols) for each df in `dfs`.
    Titles each sub-plot "Fig. {plot_idx}" and then increments plot_idx.
    Returns (fig, new_plot_idx).
    """
    n = len(dfs)
    ncols = 2
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(figsize[0], figsize[1]*nrows))
    axes = axes.flatten()

    cmap_list = [plt.cm.summer, plt.cm.spring, plt.cm.winter, plt.cm.autumn]

    for i, df in enumerate(dfs):
        ax = axes[i]
        questions[i] if questions else df.index.name or f"Plot {i+1}"
        # prepare colors, gray out "No sabe/ No contesta"
        cmap = np.random.choice(cmap_list)
        colors = cmap(np.linspace(0,0.8,len(df)))
        if 'No sabe/ No contesta' in df.index:
            gi = list(df.index).index('No sabe/ No contesta')
            colors[gi] = [0.7,0.7,0.7,1.0]


        print(f'plotting {df.index.name} ({len(df)} rows)')
        print(f'df:{df}')
        # choose horiz vs vert
        if len(df) >= 6:
            bars = ax.barh(df.index, df['%'], color=colors)
            for b in bars:
                w = b.get_width()
                ax.text(w+0.5, b.get_y()+b.get_height()/2, f"{w:.1f}%", 
                        ha="left", va="center", fontsize=8)
            ax.set_xlabel("Porcentaje (%)", fontsize=6)
            ax.invert_yaxis()
        else:
            bars = ax.bar(df.index, df['%'], color=colors)
            for b in bars:
                h = b.get_height()
                ax.text(b.get_x()+b.get_width()/2, h+0.5, f"{h:.1f}%", 
                        ha="center", va="bottom", fontsize=8)
            ax.set_ylabel("Porcentaje (%)", fontsize=6)
            ax.tick_params(axis='x', rotation=45)

        y_texts = [split_text_by_words(lbl.get_text(), n=3, k=10) for lbl in ax.get_yticklabels()]
        ax.set_yticklabels(y_texts, fontsize=8)
        x_texts = [split_text_by_words(lbl.get_text(), n=3, k=10) for lbl in ax.get_xticklabels()]
        ax.set_xticklabels(x_texts, fontsize=8)


        # set the title with current figure index
        ax.set_title(
            split_text_by_words(f"Fig. {plot_idx}\n{questions[i] if questions else df.index.name}", 8),
            fontsize=8
        )
        plot_idx += 1

    # drop any empty axes
    for extra in axes[n:]:
        fig.delaxes(extra)

    plt.tight_layout()
    return fig, plot_idx

def find_vars(v, pattern=r'([pP]\d+(?:[a-zA-Z0-9_]+)?\|[A-Z0-9]+)'):
    """
    Finds and returns all variable IDs in the string v that match the specified pattern,
    including IDs like p1_1a_1|IDE, p2_1a_1|IDE, p14a_1|IND and simpler forms like p6|IDE.

    Returns:
        list: A list of unique variable IDs found in the string.
    """
    if isinstance(v, str):
        matches = re.findall(pattern, v)
        return list(set(matches)) if matches else None
    return None

def fix_title(plot_vars, plot_idx):
    # Fix the title to be more descriptive
    tmp_title = f"Fig. {plot_idx}\n" + ', '.join([pregs_dict.get(var, 'Unknown Question') for var in plot_vars if var in pregs_dict])
    plot_idx += 1
    return tmp_title

def PATCH_df_nan_fix(df):
    """
    Fixes the DataFrame by replacing NaN values with 0 and converting to int.
    """
    tmp_df = tst_na_dict['p3_1|IDE']
    tmp_corr_df = tmp_df.loc[~tmp_df.index.isna()].copy()
    tmp_corr_df['%'] = tmp_corr_df['%'].div(tmp_corr_df['%'].sum()).mul(100)
    return tmp_corr_df

# tmp_sel_plt_dict contiene los plots seleccionados


tmp_sel_plt_dict = {st: {'df': df_tables[st]} for st in tmp_id_lst if st in df_tables}

if set(tmp_id_lst) - set(tmp_sel_plt_dict.keys()):
    missing_ids = list(set(tmp_id_lst) - set(tmp_sel_plt_dict.keys()))
    print(f"******************Warning: Some IDs were not found in df_tables: {missing_ids}****************")

else:
    print(f"**************** All IDs were found in df_tables. *******************")



ptrn_dict= {ky : structured_expert_results[ky]['EXPERT_REPLY'] for ky in structured_expert_results.keys()}

texts = {}

# 1) construcción del documento
texts = {
    'PREGUNTA: ': user_query,
    'RESPUESTA: ': final_smry_dict['QUERY_ANSWER'],
    'RESUMEN DEL ANÁLISIS': final_smry_dict['TOPIC_SUMMARY']
}

texts.update(final_smry_dict['TOPIC_SUMMARIES'])

# texts.update({'PATRONES EXPERTOS: ': ''})

# texts.update(ptrn_dict)

## PARCHE 

# parche para arreglar los nan. 
# OJO: es necesario reprocesar los resultados de la tabla para que el modelo pueda interpretarlos correctamente. Esto sólo es para los plots. 
tst_na_dict = {k: v for k, v in df_tables.items() if v.index.isna().any()}


df_tables_PTCH = df_tables.copy()
for ky in tst_na_dict.keys():
    df_tables_PTCH[ky] = PATCH_df_nan_fix(df_tables_PTCH[ky])
## ***** ARREGLAR ESTE PARCHE *****


# identificación de variables para plots
tmp_plot_dict = {ky: find_vars(v) for ky, v in texts.items()}
tmp_dfs_dict = {ky: [df_tables_PTCH[st] for st in (tmp_plot_dict[ky] or []) if st in df_tables_PTCH.keys()] for ky in tmp_plot_dict.keys()}

# prepare output dirs
os.makedirs("tmp_images", exist_ok=True)

md_lines = []

plot_idx = 1

for section, text in texts.items():
    md_lines.append(f"# {section}")
    md_lines.append(text)
    plot_vars = tmp_plot_dict.get(section) or []

    if len(plot_vars) == 1:
        var = plot_vars[0]
        df = tmp_dfs_dict.get(section, [None])[0]
        if df is not None:
            title = pregs_dict.get(var, var).split('|')[-1]
            fig = create_plot(df, f'Fig. {plot_idx}\n{title}', figsize=(5,5))
            img = f"tmp_images/{var.replace('|','_').lower()}.png"
            fig.savefig(img, dpi=150); plt.close(fig)
            md_lines.append(f"![{var}]({img})")
            plot_idx += 1

    elif len(plot_vars) > 1:
        # collect the exact dataframes and labels in order
        dfs = tmp_dfs_dict.get(section, [])
        qs  = [pregs_dict.get(var, var).split('|')[-1] for var in plot_vars]
        if dfs:
            fig, plot_idx = create_multiplot(dfs, questions=qs, figsize=(8,4))
            img = f"tmp_images/{section.replace(' ','_').lower()}_multiplot.png"
            fig.savefig(img, dpi=150)
            plt.close(fig)
            md_lines.append(f"![{section} multiplot]({img})")
            plot_idx += 1


# write out the markdown file
with open(ruta_rep + "report.md", "w") as f:
    f.write("\n\n".join(md_lines))

print("✅ report.md and images/ saved.")


# Convert to PDF using pypandoc
import pypandoc
from datetime import datetime
# Create timestamp for unique filename

# Convert list to markdown text
md_text = "\n\n".join(md_lines)

write_pdf = True

if write_pdf: 
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_pdf = f"report_{timestamp}.pdf"
    try:
        pypandoc.convert_text(
            md_text,
            to="pdf",
            format="md",
            outputfile=ruta_rep + output_pdf,
            extra_args=[
                "--pdf-engine=xelatex",
                "--variable=geometry:margin=1in",
                "--variable=mainfont:Helvetica",
                "--variable=fontsize:12pt"
            ]
        )
        print(f"PDF report saved to {output_pdf}")
    except Exception as e:
        print(f"Error converting to PDF: {str(e)}")
        print("You can try converting the markdown file manually with Pandoc or another tool.")


# TODO: arreglar pdf y lo demás....

plotting GLOBALIZACION|¿Qué tan orgulloso está de ser mexicano? (6 rows)
df:                                                        %
GLOBALIZACION|¿Qué tan orgulloso está de ser me...       
Muy orgulloso                                       55.67
Orgulloso                                           33.58
Poco orgulloso                                       8.92
Nada orgulloso                                       1.33
Otro                                                 0.08
No sabe/ No contesta                                 0.42
plotting IDENTIDAD_Y_VALORES|¿Qué tan orgulloso se siente de ser mexicano? (7 rows)
df:                                                        %
IDENTIDAD_Y_VALORES|¿Qué tan orgulloso se sient...       
Mucho                                               63.42
Algo                                                25.42
Poco                                                 8.25
Nada                                                 1.92
No soy mexicano          

/var/folders/35/kp3vmnl94d9cp_qddwwb50z00000gn/T/ipykernel_1180/1716772080.py:175: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(y_texts, fontsize=8)
/var/folders/35/kp3vmnl94d9cp_qddwwb50z00000gn/T/ipykernel_1180/1716772080.py:177: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(x_texts, fontsize=8)
/var/folders/35/kp3vmnl94d9cp_qddwwb50z00000gn/T/ipykernel_1180/1716772080.py:175: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(y_texts, fontsize=8)
/var/folders/35/kp3vmnl94d9cp_qddwwb50z00000gn/T/ipykernel_1180/1716772080.py:177: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(x_texts, 

plotting POBREZA|En su opinión, ¿cree que en México hay mexicanos de primera y segunda o que todos son iguales? (3 rows)
df:                                                        %
POBREZA|En su opinión, ¿cree que en México hay ...       
De primera y segunda                                36.92
Todos son iguales                                   60.42
No sabe/ No contesta                                 2.67
plotting INDIGENAS|¿Y cuál cree que es la mayor desventaja de ser indígena en México? (8 rows)
df:                                                        %
INDIGENAS|¿Y cuál cree que es la mayor desventa...       
Discriminación                                      45.12
Analfabetismo                                        5.42
Marginación y pobreza                               20.27
Pérdida cultural                                     1.42
Exclusión                                            4.25
Ser diferentes                                       3.42
Mención dispersa        

/var/folders/35/kp3vmnl94d9cp_qddwwb50z00000gn/T/ipykernel_1180/1716772080.py:175: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(y_texts, fontsize=8)
/var/folders/35/kp3vmnl94d9cp_qddwwb50z00000gn/T/ipykernel_1180/1716772080.py:177: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(x_texts, fontsize=8)
/var/folders/35/kp3vmnl94d9cp_qddwwb50z00000gn/T/ipykernel_1180/1716772080.py:175: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(y_texts, fontsize=8)
/var/folders/35/kp3vmnl94d9cp_qddwwb50z00000gn/T/ipykernel_1180/1716772080.py:177: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(x_texts, 

PDF report saved to report_20250526_124717.pdf


## Agente y gráfico del modelo

### LangGraph

In [ ]:
# _variable_selector() selecciona las variables, devuelve tmp_pre_res_dict y tmp_grade_dict

from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

class PatternItemGrader(BaseModel):
    GRADE_DICT: dict[float, str] = {
        0.0: "",
    }

pattern_parser_grader = PydanticOutputParser(pydantic_object=PatternItemGrader)
pattern_format_grader_instrtuctions = pattern_parser_grader.get_format_instructions()


def create_prompt_grader(user_query, tmp_svvinfo_st ,format_instructions=""):
    """
    prompt for gradig items against a query
    """
    prompt = f"""
You are an expert in survey research and in qualitative research, and you are fluent in English and Spanish. You will reply in English only.  
Your task is to read the following SURVEY INFORMATION, and a user QUERY, and then grade the SURVEY INFORMATION against the QUERY and write a one-sentence explanation of your grade.

THE SURVEY INFORMATION has 3 parts
- QUESTION: the question asked in the survey
- SUMMARY: a summary of the results of the survey
- IMPLICATIONS: a statement of the importance of the results written by an expert in the field of the survey. 

The GRADE will will be a number between 0 and 3, where: 
- 0: the QUESTION and the SUMMARY are NOT relevant to the QUERY, and the IMPLICATIONS are NOT relevant to the QUERY
- 1: the QUESTION and the SUMMARY are NOT relevant to the QUERY, but the IMPLICATIONS seem relevant to the QUERY
- 2: the QUESTION and the SUMMARY are somewhat relevant to the QUERY, but the IMPLICATIONS seem relevant to the QUERY
- 3: the QUESTION and the SUMMARY are relevant to the QUERY, and the IMPLICATIONS are relevant to the QUERY

You will write a one-sentence EXPLANATION of your grade, paying attention to how QUESTION, SUMMARY and IMPLICATIONS are related to the QUERY. Be detailed and specific in your explanation.
IMPORTANT: return an EXPLANATION regardless of the GRADE, this is, explain all your grades, even if they are 0.
IMPORTANT: make sure to match your GRADE to your EXPLANATION, this is, that they correspond to the criteria above. 

Here are some examples of the GRADE and the explanation:
- EXAMPLE_QUERY : "Would selling strawberry ice cream is a good idea?"

- EXAMPLE_SURVEY_INFORMATION_1 : "QUESTION: Do you like strawberry ice cream? SUMMARY: 50% of people like strawberry ice cream. IMPLICATIONS: Selling strawberry ice cream is a good idea."
- EXAMPLE_GRADE_1 : 3
- EXAMPLE_EXPLANATION_1 : "The QUESTION is relevant to the QUERY because it asks about strawberry ice cream, the SUMMARY is relevant because it provides information about people's preferences, and the IMPLICATIONS are relevant because they suggest that selling strawberry ice cream is a good idea."

- EXAMPLE_SURVEY_INFORMATION_2 : "QUESTION: Do you like chocolate ice cream? SUMMARY: 50% of people like chocolate ice cream. IMPLICATIONS: Selling chocolate ice cream is a good idea."  
- EXAMPLE_GRADE_2 : 2
- EXAMPLE_EXPLANATION_2 : "The QUESTION is not relevant to the QUERY because it asks about chocolate ice cream, but the SUMMARY is relevant because it provides information about people's preferences, and the IMPLICATIONS are relevant because they suggest that selling chocolate ice cream is a good idea for alternate flavors."

- EXAMPLE_SURVEY_INFORMATION_3 : "QUESTION: Have you seen the movie 'Wild Strawberries' by Ingmar Bergman? SUMMARY: 50% of people have seen the movie. IMPLICATIONS: The movie is a classic and is worth watching."
- EXAMPLE_GRADE_3 : 1
- EXAMPLE_EXPLANATION_3 : "The QUESTION is not relevant to the QUERY because it asks about a movie, and the SUMMARY is not relevant because it talks about movies, not ice cream, and the IMPLICATIONS are seem relevant because they talk about a movie with 'ice cream' in the title."

- EXAMPLE_SURVEY_INFORMATION_4 : "QUESTION: How many times have you gone to the beach? SUMMARY: 50% of people go to the beach once a year. IMPLICATIONS: The beach is a popular destination."
- EXAMPLE_GRADE_4 : 0
- EXAMPLE_EXPLANATION_4 : "The QUESTION is not relevant to the QUERY because it asks about going to the beach, and the SUMMARY is not relevant because it talks about the beach, not ice cream, and the IMPLICATIONS are not relevant because they talk about a beach, not ice cream."

Example output (strict JSON, no markdown, no code block, no extra text).
{{
  GRADE: EXPLANATION
}}

QUERY: {user_query}
SURVEY_INFORMATION: {tmp_svvinfo_st}

{format_instructions}

Checklist before submitting:
- [ ] GRADE has been calculated.
- [ ] EXPLANATION has been written.
- [ ] No field is left empty.
"""
    return prompt

def get_structured_summary_grader_p(prompt, model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    # prompt =create_prompt_grader(user_query=user_query, tmp_svvinfo_st= tmp_svvinfo_st, format_instructions=pattern_format_grader_instrtuctions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_grader.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_grader.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}
    
def create_tmp_svyinfo_dict(tmp_ky, top_ids, tmp_pre_res_dict):
    """         
    Create a temporary survey information dictionary for a given key.
    Args:
        tmp_ky (int): The index of the key in the top_ids list.
        top_ids (list): The list of top IDs.
        tmp_pre_res_dict (dict): The dictionary containing preprocessed results.
    Returns:

        dict: A dictionary containing the survey information for the specified key.
    """
    # Create a temporary survey information dictionary for a given key
    tmp_id_st = top_ids[tmp_ky]
    tmp_svyinfo_dict = {k.split('__')[1].upper(): v for k,v in tmp_pre_res_dict.items() if k.startswith(tmp_id_st)}
    tmp_svvinfo_st = ' '.join([f'{k}: {v}' for k,v in tmp_svyinfo_dict.items()])
    return tmp_svvinfo_st

def batch_process_expert_grader(user_query, top_ids, tmp_pre_res_dict, model_name , batch_size=8192):
    """
    Processes expert summaries in batches, saving checkpoints after each batch.

    Parameters:
        top_ids (list): List of variables to grade.
        tmp_pre_res_dict (dict): Dict with survey results.
        batch_size (int): Maximum token limit for each batch.

    Returns:
        dict: A dictionary of structured results.
    """


    # Prepare prompts and keys
    prompts = [
        create_prompt_grader(user_query,  
                             create_tmp_svyinfo_dict(tmp_ky, top_ids, 
                                                     tmp_pre_res_dict),
                                                     format_instructions=pattern_format_grader_instrtuctions)
        for tmp_ky in range(len(top_ids))
    ]
    keys = top_ids
    
    # Batch the documents
    batches = batch_documents(prompts, keys, max_tokens=batch_size, encoding_name="cl100k_base")
    
    # Initialize results dictionary
    structured_results = {}
    
    # Process each batch
    with tqdm.tqdm(total=len(keys), desc="Processing Expert Summaries") as pbar:
        for batch_docs, batch_keys in batches:
            for prompt, key in zip(batch_docs, batch_keys):
                try:
                    _, tmp_grade_dict = get_structured_summary_grader_p(prompt, model_name = model_name, temperature= 0.5)
                    #print(f'tmp_grade_dict: {tmp_grade_dict}')
                    structured_results[key] = tmp_grade_dict
                except Exception as e:
                    structured_results[key] = {'error': str(e)}
                pbar.update(1)
    
    return structured_results

def _variable_selector(user_query, top_vals=30):
    """
    Selects the top variables based on their relevance to the user query.
    Returns:
        dict: A dictionary of selected variables with their grades.
    """
    # Turn it into a vector
    print("Embedding the user query...")
     # embedding_fun_openai is defined above
    query_emb = embedding_fun_openai([user_query])[0]

    # tmp_dist_df contiene las distancias para los tres tipos/ facets, normalizadas entre 0 y 1

    # OJO: esto obviamente asume que los tres tipos de información tienen la misma importancia, lo cual no es inicialmente cierto.
    # Pero la mezcla de las tres calificaciones devuelve una variedad más amplia de resultados.  

    type_lst = [ "question", "summary", "implications"]

    tmp_dist_dict = {}
    for type in type_lst:
        print(f"Querying for type: {type}")
        tmp_res_q = db_f1.query(
            query_embeddings = [query_emb],
            n_results        = 100,  # devuelvo 100 resultados para cada tipo con distancias menores
            where            = {"type": type}
        )
        [tmp_res_ids] = tmp_res_q['ids']
        [tmp_res_distances] = tmp_res_q['distances']

        tmp_dist_dict[type]= dict(zip(tmp_res_ids, tmp_res_distances))

    # remove the suffixes from the keys
    tmp_dist_dict = { outer_key: { k.split('__')[0]: v for k, v in inner_dict.items() }
        for outer_key, inner_dict in tmp_dist_dict.items() }

    # Create a DataFrame where keys in every subdict are the index and keys in tmp_dist_dict are columns
    tmp_dist_df = pd.DataFrame.from_dict(tmp_dist_dict)

    # Normalize each column so that max = 1 and min = 0
    tmp_dist_df = (tmp_dist_df - tmp_dist_df.min()) / (tmp_dist_df.max() - tmp_dist_df.min())


    tmp_dist_df['mean'] = tmp_dist_df.mean(axis=1)
    tmp_dist_df.sort_values(by='mean', ascending=True, inplace=True)

    top_ids = tmp_dist_df.head(top_vals).index.tolist()

    tmp_list = []

    for type in type_lst:
        for id in top_ids:
            tmp_list.append(id + f'__{type}')


    # Retrieve documents using the list of ids
    result_by_ids = db_f1.get(ids=tmp_list)

    tmp_pre_res_dict = dict(zip(result_by_ids['ids'], result_by_ids['documents']))

    ## generación del esquema de pydantic para la respuesta del evaluador de relevancia


    tst_res = batch_process_expert_grader(user_query, top_ids, tmp_pre_res_dict, mod_alto, batch_size=8192)
    tmp_grade_dict=  {k: v['GRADE_DICT'] for k, v in tst_res.items()}

    # Filtrar los elementos que tienen una calificación mayor a 1

    tmp_grade_dict= {k: v for k, v in tmp_grade_dict.items() if list(v.keys())[0] >1 }

    return tmp_pre_res_dict, tmp_grade_dict



tmp_pre_res_dict, tmp_grade_dict

Embedding the user query...
Querying for type: question
Querying for type: summary
Querying for type: implications


Processing Expert Summaries: 100%|██████████| 30/30 [01:02<00:00,  2.08s/it]


{'p5|CUL': {3.0: 'The QUESTION is relevant to the QUERY because it asks about feelings of pride in being Mexican, the SUMMARY is relevant because it provides data on how Mexicans perceive their national identity, and the IMPLICATIONS are relevant because they discuss how these feelings of pride relate to cultural and political identity, directly addressing what it means to be Mexican for Mexicans.'},
 'p6|IDE': {3.0: 'The QUESTION is relevant to the QUERY because it directly explores feelings of pride in being Mexican, the SUMMARY is relevant because it quantifies those feelings among respondents, and the IMPLICATIONS are relevant because they discuss how these insights inform understandings of Mexican identity and can be used to shape related policies and initiatives.'},
 'p1_1a_1|IDE': {3.0: "The QUESTION is relevant to the QUERY because it asks respondents for words they associate with 'México', directly probing perceptions of Mexican identity, the SUMMARY is relevant because it det

In [ ]:
# TODO: identificar tipo de reporte: mínimo descriptivo (un párrafo de resumen y gráficas) o temático (varios párrafos de resumen y gráficas por tema).
# TODO: reconstruir lista de vars seleccionadas con base en retro de usuario - si no la lay, pasar tmp_grade_dict

In [ ]:
# _deep_analyzer() genera el análisis tranversal de los expertos 


from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser
from typing import Dict
from pydantic import BaseModel
import tqdm



class PatternItem(BaseModel):
    TITLE_SUMMARY: str
    VARIABLE_STRING: str
    DESCRIPTION: str

class FlatPatternSummary(BaseModel):
    """
    A “catch-all” model so we can declare SIMILAR_PATTERN_1…n
    and DIFFERENT_PATTERN_1…n at the top level.
    """
    class Config:
        extra = "allow"

class ExpertSummaryResponse(BaseModel):
    EXPERT_REPLY: str


class TransversalAnalysisResponse(BaseModel):
    TOPIC_SUMMARIES: dict[str, str]  # A dictionary with topic names as keys and summaries as values
    TOPIC_SUMMARY: str  # A one-paragraph summary for a general audience
    QUERY_ANSWER: str  # A two-sentence answer to the query


def flat_pattern_template(n: int) -> FlatPatternSummary:
    """
    Build an “empty” flat summary with n SIMILAR_PATTERN_i
    and n DIFFERENT_PATTERN_i placeholders.
    """
    empty = {"TITLE_SUMMARY": "", "VARIABLE_STRING": "", "DESCRIPTION": ""}
    payload: dict[str, dict[str, str]] = {}
    for kind in ("SIMILAR_PATTERN", "DIFFERENT_PATTERN"):
        for i in range(1, n + 1):
            payload[f"{kind}_{i}"] = empty.copy()
    return FlatPatternSummary(**payload)

def create_prompt_crosssum(user_query, tmp_res_st, n_topics=5, format_instructions=""):
    """
    Optimized prompt for extracting non-empty, detailed patterns from survey results.
    """
    prompt = f"""
You are a research assistant analyzing survey results to answer the QUERY below.

Your job is to:
- Identify the top {n_topics} SIMILAR PATTERNS (trends or agreements) and {n_topics} DIFFERENT PATTERNS (contrasts or contradictions) relevant to the QUERY.
- For each pattern, provide:
    1. TITLE_SUMMARY: A short, descriptive title (never empty).
    2. VARIABLE_STRING: Comma-separated QUESTION_IDs used in the pattern (never empty).
    3. DESCRIPTION: A detailed explanation, citing numbers and QUESTION_IDs in parentheses (never empty).

Instructions:
- Do NOT leave any field empty. If information is limited, generalize or summarize what is available.
- Use only facts you are sure about, and always cite the QUESTION_ID for each fact.
- Do NOT repeat the same pattern in multiple fields.
- Ignore any results marked as 'NaN' or not available.
- If you combine categories (e.g., "like a lot" + "like somewhat"), only mention the sum if all original values are present.
- Do NOT invent data; if a pattern is weak, explain that.
- Each pattern must be unique and relevant to the QUERY.

Example input format:
* QUESTION_ID: p1_1|ABC
  QUESTION: 'Do you like ice cream?'
  SUMMARY: 85% of people like ice cream, 10% do not, 5% do not know.
* QUESTION_ID: p2_1|ABC
  QUESTION: 'Do you like marshmallow treats?'
  SUMMARY: 80% like, 15% do not, 5% do not know.
* QUESTION_ID: p10_1|DEF:
  QUESTION: 'Do you like sour apple candies?'
  SUMMARY: 40% of people like sour apple candies, while 50% said they do not like it, and 10% said they do not know.

In these cases, the SIMILAR PATTERN is that 'people really like sweet treats because 85% like ice cream and 80% like marshmallow treats', 
and the DIFFERENT PATTERN is that 'people do not like sour treats as much because only 40% said they liked them'.

Example output for a SIMILAR PATTERN:
TITLE_SUMMARY: High preference for sweet treats
VARIABLE_STRING: p1_1|ABC,p2_1|ABC
DESCRIPTION: A large majority like ice cream (85%, p1_1|ABC) and marshmallow treats (80%, p2_1|ABC).

Note that the QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where xx is any combination of numbers and '_', and YYY are any three capital letters). 
There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


Example output (strict JSON, no markdown, no code block, no extra text):
{{
  "SIMILAR_PATTERN_1": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}},
  "SIMILAR_PATTERN_2": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}},
  ...
  "DIFFERENT_PATTERN_5": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}}
}}

QUERY: {user_query}
RESULTS: {tmp_res_st}

{format_instructions}

Checklist before submitting:
- [ ] All fields are filled for each pattern.
- [ ] Each DESCRIPTION includes at least one number and QUESTION_ID.
- [ ] No field is left empty.
"""
    return prompt

def get_structured_summary(user_query=user_query, tmp_res_st=[], model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.

    # Pedir n patrones: tres resultados mínimos para cada patrón -- para minimimzar alucinaciones. 
    n_topics = min(len(tmp_grade_dict) // 4, 5)

    prompt =create_prompt_crosssum(user_query=user_query, tmp_res_st=tmp_res_st, n_topics=n_topics, format_instructions=pattern_simdif_format_instructions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_simdif.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_simdif.parse(cleaned)
        return parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}

def create_prompt_expt_smry(tst_lgc_dict, tmp_ky, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and survey summaries.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """

    ky = tmp_ky

    # Variables identified by the model
    tst_str_lst = tst_lgc_dict[ky]['VARIABLE_STRING'].split(',')
    tst_str_lst = [st + '__question' for st in tst_str_lst]
    print(f'Variables identified by the model: {tst_str_lst}')

    # Variables in the database
    tmp_db_var_lst = db_f1.get(ids=tst_str_lst)['ids']
    tmp_db_var_lst = list(set(tmp_db_var_lst))
    print(f'Variables in the database: {tmp_db_var_lst}')

    # Hallucinated variables
    tmp_hlc_var_lst = set(tst_str_lst) - set(tmp_db_var_lst)
    if tmp_hlc_var_lst:
        print(f'Hallucinated variables by the model: {tmp_hlc_var_lst}')

    # Implications for the identified variables
    tmp_imp_lst = [st.replace('question', 'implications') for st in tmp_db_var_lst]
    tmp_ky_lst = [st.split('__')[0] for st in tmp_imp_lst]
    tmp_imp_dict = dict(
        zip(
            tmp_ky_lst,
            db_f1.get(ids=tmp_imp_lst)['documents']
        )
    )

    imp_st = ' * '.join(tmp_imp_dict.values())
    tmp_smry = tst_lgc_dict[ky]['DESCRIPTION']

    # Generate the prompt
    prompt = f"""
    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do your work in either language. But you will reply in English.
    
    Your task is to read the EXPERT STATEMENTS from one or more experts about what information they consider important to receive from the survey results. 
    Then you will read a SURVEY SUMMARY of some survey results and will write a one-paragraph reply to the experts, elaborating on how the results relate and illustrate their concerns. 
    Note that they are experts and expect a detailed and thorough reply. Reply to all of them in the same paragraph, but be sure to address all of their points. 
    Do not refer to the experts or to yourself in the first person, just write the paragraph as if it was a report.

    Note that the SURVEY SUMMARY will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    They are not relevant to your answer, but you will need to keep track of these QUESTION_IDS of format pxx|YYY in your RETURN OBJECT.
    There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    EXPERT STATEMENTS: * {imp_st}
    SURVEY SUMMARY: {tmp_smry}

    {format_instructions}
    """
    return prompt

def get_structured_expert_summary(tst_lgc_dict, tmp_ky, model_name="gpt-4o-mini-2024-07-18", temperature=0.9):
    """
    Generates a structured expert summary for a given key in the logical group dictionary.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        model_name (str): Name of the LLM model to use.
        temperature (float): Temperature setting for the LLM.

    Returns:
        dict: Parsed response as a dictionary.
    """
    # Generate the prompt
    prompt = create_prompt_expt_smry(
        tst_lgc_dict=tst_lgc_dict,
        tmp_ky=tmp_ky,
        format_instructions=expert_summary_format_instructions
    )
    
    # Get the response from the model
    response = get_answer(prompt=prompt, model=model_name, temperature=temperature)
    
    # Parse the response
    parsed = expert_summary_parser.parse(response)
    
    # Ensure the parsed response is returned as a dictionary
    return parsed.model_dump()

def batch_process_expert_summaries(tst_lgc_dict, batch_size=8192):
    """
    Processes expert summaries in batches, saving checkpoints after each batch.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        batch_size (int): Maximum token limit for each batch.

    Returns:
        dict: A dictionary of structured results.
    """
    # Prepare prompts and keys
    prompts = [
        create_prompt_expt_smry(tst_lgc_dict, key, format_instructions=expert_summary_format_instructions)
        for key in tst_lgc_dict.keys()
    ]
    keys = list(tst_lgc_dict.keys())
    
    # Batch the documents
    batches = batch_documents(prompts, keys, max_tokens=batch_size, encoding_name="cl100k_base")
    
    # Initialize results dictionary
    structured_results = {}
    
    # Process each batch
    with tqdm.tqdm(total=len(keys), desc="Processing Expert Summaries") as pbar:
        for batch_docs, batch_keys in batches:
            for prompt, key in zip(batch_docs, batch_keys):
                try:
                    # Call get_structured_expert_summary for each key
                    structured_results[key] = get_structured_expert_summary(
                        tst_lgc_dict=tst_lgc_dict,
                        tmp_ky=key,
                        model_name="gpt-4o-mini-2024-07-18",
                        temperature=0.9
                    )
                except Exception as e:
                    structured_results[key] = {'error': str(e)}
                pbar.update(1)

    
    return structured_results

def create_prompt_trnsvl(tmp_smry_st, user_query, n_cmn_tpc, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and answering a query.

    Parameters:
        tmp_smry_st (str): The list of expert statements.
        user_query (str): The query to be answered.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """
    prompt = f"""
    You are a very thorough research assistant and expert in survey research and public opinion.
    You are fully bilingual in English and Spanish, and can do your work in either language.

    Your task is to perform three analyses and return a single Python dictionary with the results.

    1) Read the following list of STATEMENTS made by experts in several topics, which mention results, their implications, and their relevance.
    Each statement starts with the marker `*` and contains all information for a single statement: results, implications, and relevance.
    Identify {n_cmn_tpc} common topics across the STATEMENTS and write a one-paragraph summary of each topic, citing the most relevant numbers and explanations for each.
    Format your answer as a Python dictionary with the names of the topics as keys in ALL CAPS in the same language as the query, and the summaries as values.

    Note that the STATEMENTS will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.
    They are not relevant to your answer and your will use them only to identify the variables in the statements that your will use to write the summaries.

    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    Here are the statements: {tmp_smry_st}

    Save your summaries in a Python dictionary with the key `TOPIC_SUMMARIES`.

    2) Read your `TOPIC_SUMMARIES` and write a one-paragraph summary of the most relevant results and implications of the survey results, written for a general audience.
    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).
    Save your summary in a Python dictionary with the key `TOPIC_SUMMARY`.

    3) Read the QUERY and your `TOPIC_SUMMARY` and write a two-sentence answer to the QUERY that summarizes the most important points of your `TOPIC_SUMMARY`.
    Do not include any numbers or facts in your answer, just your reply to the QUERY.
    Be concise and do not repeat numbers; just answer the QUERY with the relevant ideas.
    
    Here is the QUERY: {user_query}

    Save your answer in a Python dictionary with the key `QUERY_ANSWER`.

    IMPORTANT: Your replies for all three tasks must be in the language of the QUERY.
    IMPORTANT: Make sure to return only a correctly formatted Python dictionary, without any code block formatting, markdown, or additional text.

    {format_instructions}
    """
    return prompt

def get_transversal_analysis(tmp_smry_st, user_query, model_name='gpt-4o-mini-2024-07-18', temperature=0.9):
    # Generate the prompt
    prompt = create_prompt_trnsvl(
        tmp_smry_st=tmp_smry_st,
        user_query=user_query,
        n_cmn_tpc=3,
        format_instructions=transversal_format_instructions
    )
    
    # Get the response from the model
    response = get_answer(prompt=prompt, model=model_name, temperature=temperature)
    parsed = transversal_parser.parse(response)
    # Parse the response
    return parsed.model_dump()  # Returns the parsed response as a dictionary.

# Parsers

pattern_parser_simdif = PydanticOutputParser(pydantic_object=FlatPatternSummary)
pattern_simdif_format_instructions = pattern_parser_simdif.get_format_instructions()

expert_summary_parser = PydanticOutputParser(pydantic_object=ExpertSummaryResponse)
expert_summary_format_instructions = expert_summary_parser.get_format_instructions()

transversal_parser = PydanticOutputParser(pydantic_object=TransversalAnalysisResponse)
transversal_format_instructions = transversal_parser.get_format_instructions()



def _deep_analyzer(tmp_pre_res_dict, tmp_grade_dict):
    """
    internal function to produce the transversal analysis and expert summaries
    Args: tmp_pre_res_dict (dict): Dictionary containing preprocessed results.
    Returns:
        tuple: A tuple containing two dictionaries:
        - tmp_preproc_dic: Filtered dictionary with relevant questions and summaries.
        - final_smry_dict: Dictionary with the final transversal analysis.
        - structured_expert_results: Dictionary with structured expert summaries.
    """

    tmp_preproc_dic = {k: v for k, v in tmp_pre_res_dict.items() if k.split('__')[1] in ['question', 'summary'] }

    # filtro de las preguntas relevantes
    tmp_preproc_dic ={k: v for k, v in tmp_preproc_dic.items() if any(k.startswith(grade_key) for grade_key in tmp_grade_dict.keys())}

    tmp_combined_strings = []

    for i, (k, v) in enumerate(tmp_preproc_dic.items(), start=1):
        facet = k.split("__", 1)[1].upper()
        p_id = k#.split("__")[0].split("|")[0]

        grouped_index = (i + 1) // 2 
        # q_id = v.split("|")[0]
        parts = v.split("|", 1)
        text = parts[1] if len(parts) > 1 else parts[0]

        p_id = p_id.split("__")[0]

        tmp_combined_strings.append(f"{facet}_{grouped_index}|{p_id}: {text}")

    tmp_res_st = '\n'.join(tmp_combined_strings)

    tst_lgc_dict = get_structured_summary(user_query=user_query, tmp_res_st=tmp_res_st, model_name = mod_alto, temperature= 0.0)

    structured_expert_results = batch_process_expert_summaries(tst_lgc_dict)

    tmp_smry_st = ' * '.join([v['EXPERT_REPLY'] for v in structured_expert_results.values()])

    final_smry_dict = get_transversal_analysis(tmp_smry_st, user_query)
    return tmp_preproc_dic, final_smry_dict, structured_expert_results


In [ ]:
# _report_writer() crea el reporte final en PDF

import numpy as np
import math
import matplotlib.pyplot as plt
import pypandoc
from datetime import datetime

write_pdf = True


def split_text_by_words(text, n=8, k=50):
    """
    Splits a string into lines with a maximum of n words per line.
    If the first n-1 words already exceed k characters, truncates with '...'.
    """
    words = text.split()
    # if first n-1 words already too long → truncate
    first_chunk = words[: n-1]
    if len(" ".join(first_chunk)) >= k:
        return " ".join(first_chunk) + "..."
    # otherwise chunk every n words
    lines = []
    for i in range(0, len(words), n):
        chunk = words[i : i + n]
        lines.append(" ".join(chunk))
    return "\n".join(lines)

def create_plot(df, question, figsize=(6, 8)):
    """
    Creates a bar plot (horizontal if rows <= 6, vertical otherwise) of the survey data.

    Args:
        df (pd.DataFrame): DataFrame containing the data to plot.
        question (str): The survey question to use as the plot title.

    Returns:
        matplotlib.pyplot.figure: The matplotlib figure object.
    """
    # Determine plot type based on the number of rows
    if len(df) >= 6:
        # Horizontal bar plot
        fig, ax = plt.subplots(figsize=figsize)
        
        # Create a colormap where 'No sabe/ No contesta' is always gray

        colormaps = [plt.cm.summer, plt.cm.spring, plt.cm.winter, plt.cm.autumn]
        selected_colormap = np.random.choice(colormaps)
        colors = selected_colormap(np.linspace(0, 0.8, len(df)))

        if 'No sabe/ No contesta' in df.index:
            gray_index = list(df.index).index('No sabe/ No contesta')
            colors[gray_index] = [0.7, 0.7, 0.7, 1.0]  # RGBA for gray
        bars = ax.barh(df.index, df['%'], color=colors)

        # Add percentage labels to the bars
        for bar in bars:
            width = bar.get_width()
            label_position = width + 0.5  # Slightly to the right of the bar
            ax.text(label_position, bar.get_y() + bar.get_height()/2,
                    f'{width:.1f}%',
                    ha='left', va='center', fontsize=8)
        # Customize plot
        ax.set_xlabel('Porcentaje (%)', fontsize=8)
        ax.set_title(split_text_by_words(question), fontsize=9)
        # Apply split_text_by_words to y-tick labels with conditional fontsize
        y_labels = [split_text_by_words(label, n=8, k=50) for label in df.index]
        y_fontsizes = [10 if len(label.split()) <= 4 else 8 for label in df.index]
        ax.set_yticklabels(y_labels)
        for label, fontsize in zip(ax.get_yticklabels(), y_fontsizes):
            label.set_fontsize(fontsize)

        # Apply split_text_by_words to x-tick labels with conditional fontsize
        x_labels = [split_text_by_words(str(label), n=8, k=50) for label in ax.get_xticks()]
        x_fontsizes = [10 if len(str(label).split()) <= 4 else 8 for label in ax.get_xticks()]
        ax.set_xticklabels(x_labels)
        for label, fontsize in zip(ax.get_xticklabels(), x_fontsizes):
            label.set_fontsize(fontsize)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Invert the y-axis to show the first item at the top
        ax.invert_yaxis()
        plt.tight_layout()

    else:
        # Vertical bar plot
        fig, ax = plt.subplots(figsize=figsize)

        colormaps = [plt.cm.summer, plt.cm.spring, plt.cm.winter, plt.cm.autumn]
        selected_colormap = np.random.choice(colormaps)
        colors = selected_colormap(np.linspace(0, 0.8, len(df)))

        if 'No sabe/ No contesta' in df.index:
            gray_index = list(df.index).index('No sabe/ No contesta')
            colors[gray_index] = [0.5, 0.5, 0.5, 1.0]  # RGBA for gray

        bars = ax.bar(df.index, df['%'], color=colors)

        # Add percentage labels above the bars
        for bar in bars:
            height = bar.get_height()
            label_position = height + 0.5
            ax.text(bar.get_x() + bar.get_width()/2, label_position,
                    f'{height:.1f}%',
                    ha='center', va='bottom', fontsize=8)

        # Customize plot
        ax.set_ylabel('Porcentaje (%)', fontsize=8)

        # Apply split_text_by_words to y-tick labels with conditional fontsize
        y_labels = [split_text_by_words(label, n=5, k=40) for label in df.index]
        y_fontsizes = [10 if len(label.split()) <= 4 else 6 for label in df.index]
        ax.set_yticklabels(y_labels)
        for label, fontsize in zip(ax.get_yticklabels(), y_fontsizes):
            label.set_fontsize(fontsize)

        # Apply split_text_by_words to x-tick labels with conditional fontsize
        x_labels = [split_text_by_words(str(label), n=8, k=50) for label in ax.get_xticks()]
        x_fontsizes = [10 if len(str(label).split()) <= 4 else 8 for label in ax.get_xticks()]
        ax.set_xticklabels(x_labels)
        for label, fontsize in zip(ax.get_xticklabels(), x_fontsizes):
            label.set_fontsize(fontsize)
        ax.set_title(split_text_by_words(question, 8), fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()

    return fig

def create_multiplot(dfs, questions=None, plot_idx=1, figsize=(10,5)):
    """
    Creates a grid of subplots (2 cols) for each df in `dfs`.
    Titles each sub-plot "Fig. {plot_idx}" and then increments plot_idx.
    Returns (fig, new_plot_idx).
    """
    n = len(dfs)
    ncols = 2
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(figsize[0], figsize[1]*nrows))
    axes = axes.flatten()

    cmap_list = [plt.cm.summer, plt.cm.spring, plt.cm.winter, plt.cm.autumn]

    for i, df in enumerate(dfs):
        ax = axes[i]
        questions[i] if questions else df.index.name or f"Plot {i+1}"
        # prepare colors, gray out "No sabe/ No contesta"
        cmap = np.random.choice(cmap_list)
        colors = cmap(np.linspace(0,0.8,len(df)))
        if 'No sabe/ No contesta' in df.index:
            gi = list(df.index).index('No sabe/ No contesta')
            colors[gi] = [0.7,0.7,0.7,1.0]


        print(f'plotting {df.index.name} ({len(df)} rows)')
        print(f'df:{df}')
        # choose horiz vs vert
        if len(df) >= 6:
            bars = ax.barh(df.index, df['%'], color=colors)
            for b in bars:
                w = b.get_width()
                ax.text(w+0.5, b.get_y()+b.get_height()/2, f"{w:.1f}%", 
                        ha="left", va="center", fontsize=8)
            ax.set_xlabel("Porcentaje (%)", fontsize=6)
            ax.invert_yaxis()
        else:
            bars = ax.bar(df.index, df['%'], color=colors)
            for b in bars:
                h = b.get_height()
                ax.text(b.get_x()+b.get_width()/2, h+0.5, f"{h:.1f}%", 
                        ha="center", va="bottom", fontsize=8)
            ax.set_ylabel("Porcentaje (%)", fontsize=6)
            ax.tick_params(axis='x', rotation=45)

        y_texts = [split_text_by_words(lbl.get_text(), n=3, k=10) for lbl in ax.get_yticklabels()]
        ax.set_yticklabels(y_texts, fontsize=8)
        x_texts = [split_text_by_words(lbl.get_text(), n=3, k=10) for lbl in ax.get_xticklabels()]
        ax.set_xticklabels(x_texts, fontsize=8)


        # set the title with current figure index
        ax.set_title(
            split_text_by_words(f"Fig. {plot_idx}\n{questions[i] if questions else df.index.name}", 8),
            fontsize=8
        )
        plot_idx += 1

    # drop any empty axes
    for extra in axes[n:]:
        fig.delaxes(extra)

    plt.tight_layout()
    return fig, plot_idx

def find_vars(v, pattern=r'([pP]\d+(?:[a-zA-Z0-9_]+)?\|[A-Z0-9]+)'):
    """
    Finds and returns all variable IDs in the string v that match the specified pattern,
    including IDs like p1_1a_1|IDE, p2_1a_1|IDE, p14a_1|IND and simpler forms like p6|IDE.

    Returns:
        list: A list of unique variable IDs found in the string.
    """
    if isinstance(v, str):
        matches = re.findall(pattern, v)
        return list(set(matches)) if matches else None
    return None

def fix_title(plot_vars, plot_idx):
    # Fix the title to be more descriptive
    tmp_title = f"Fig. {plot_idx}\n" + ', '.join([pregs_dict.get(var, 'Unknown Question') for var in plot_vars if var in pregs_dict])
    plot_idx += 1
    return tmp_title

def PATCH_df_nan_fix(df):
    """
    Fixes the DataFrame by replacing NaN values with 0 and converting to int.
    """
    tmp_df = tst_na_dict['p3_1|IDE']
    tmp_corr_df = tmp_df.loc[~tmp_df.index.isna()].copy()
    tmp_corr_df['%'] = tmp_corr_df['%'].div(tmp_corr_df['%'].sum()).mul(100)
    return tmp_corr_df


# extracción de plots

def _report_writer(
    user_query, 
    final_smry_dict, 
    structured_expert_results, 
    df_tables, 
    pregs_dict, 
    ruta_rep="reports/",
    write_pdf=True
):
    """
    Writes a report based on the final summary dictionary and expert results.
    """
    
    # Extract unique IDs from the final summary dictionary

    tmp_txt_lst = [final_smry_dict['TOPIC_SUMMARY']] + [st for st in final_smry_dict['TOPIC_SUMMARIES'].values()]

    # Update pattern to match expanded format including letters in ID
    # pattern = r'\(([pP]\d+[a-zA-Z]?(?:_\d+)?(?:\|[A-Z0-9]+))\)'

    # Extract matches from each text in tmp_txt_lst
    tmp_matches = []
    for text in tmp_txt_lst:
        matches = find_vars(text) #, pattern)
        if matches:
            tmp_matches.extend(matches)

    # Remove duplicates while preserving order
    tmp_id_lst = list(dict.fromkeys(tmp_matches))

    # variables en los resultados que sí existen entre las que fueron pasadas al modelo:
    print(f"Found {len(tmp_id_lst)} unique question IDs: {tmp_id_lst}")


    tmp_sel_plt_dict = {st: {'df': df_tables[st]} for st in tmp_id_lst if st in df_tables}

    if set(tmp_id_lst) - set(tmp_sel_plt_dict.keys()):
        missing_ids = list(set(tmp_id_lst) - set(tmp_sel_plt_dict.keys()))
        print(f"******************Warning: Some IDs were not found in df_tables: {missing_ids}****************")

    else:
        print(f"**************** All IDs were found in df_tables. *******************")



    ptrn_dict= {ky : structured_expert_results[ky]['EXPERT_REPLY'] for ky in structured_expert_results.keys()}

    texts = {}

    # 1) construcción del documento
    texts = {
        'PREGUNTA: ': user_query,
        'RESPUESTA: ': final_smry_dict['QUERY_ANSWER'],
        'RESUMEN DEL ANÁLISIS': final_smry_dict['TOPIC_SUMMARY']
    }

    texts.update(final_smry_dict['TOPIC_SUMMARIES'])


    ## ******** PARCHE ************* 

    # parche para arreglar los nan. 
    # OJO: es necesario reprocesar los resultados de la tabla para que el modelo pueda interpretarlos correctamente. Esto sólo es para los plots. 
    tst_na_dict = {k: v for k, v in df_tables.items() if v.index.isna().any()}


    df_tables_PTCH = df_tables.copy()
    for ky in tst_na_dict.keys():
        df_tables_PTCH[ky] = PATCH_df_nan_fix(df_tables_PTCH[ky])
    ## ***** ARREGLAR ESTE PARCHE *****


    # identificación de variables para plots
    tmp_plot_dict = {ky: find_vars(v) for ky, v in texts.items()}
    tmp_dfs_dict = {ky: [df_tables_PTCH[st] for st in (tmp_plot_dict[ky] or []) if st in df_tables_PTCH.keys()] for ky in tmp_plot_dict.keys()}

    # prepare output dirs
    os.makedirs("tmp_images", exist_ok=True)

    md_lines = []

    plot_idx = 1

    for section, text in texts.items():
        md_lines.append(f"# {section}")
        md_lines.append(text)
        plot_vars = tmp_plot_dict.get(section) or []

        if len(plot_vars) == 1:
            var = plot_vars[0]
            df = tmp_dfs_dict.get(section, [None])[0]
            if df is not None:
                title = pregs_dict.get(var, var).split('|')[-1]
                fig = create_plot(df, f'Fig. {plot_idx}\n{title}', figsize=(5,5))
                img = f"tmp_images/{var.replace('|','_').lower()}.png"
                fig.savefig(img, dpi=150); plt.close(fig)
                md_lines.append(f"![{var}]({img})")
                plot_idx += 1

        elif len(plot_vars) > 1:
            # collect the exact dataframes and labels in order
            dfs = tmp_dfs_dict.get(section, [])
            qs  = [pregs_dict.get(var, var).split('|')[-1] for var in plot_vars]
            if dfs:
                fig, plot_idx = create_multiplot(dfs, questions=qs, figsize=(5,5))
                img = f"tmp_images/{section.replace(' ','_').lower()}_multiplot.png"
                fig.savefig(img, dpi=150)
                plt.close(fig)
                md_lines.append(f"![{section} multiplot]({img})")
                plot_idx += 1


    # write out the markdown file
    with open(ruta_rep + "report.md", "w") as f:
        f.write("\n\n".join(md_lines))

    print("✅ report.md and images/ saved.")

    if write_pdf: 
        # Convert list to markdown text
        md_text = "\n\n".join(md_lines)

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_pdf = f"report_{timestamp}.pdf"
        try:
            pypandoc.convert_text(
                md_text,
                to="pdf",
                format="md",
                outputfile=ruta_rep + output_pdf,
                extra_args=[
                    "--pdf-engine=xelatex",
                    "--variable=geometry:margin=1in",
                    "--variable=mainfont:Helvetica",
                    "--variable=fontsize:12pt"
                ]
            )
            print(f"PDF report saved to {output_pdf}")
        except Exception as e:
            print(f"Error converting to PDF: {str(e)}")
            print("You can try converting the markdown file manually with Pandoc or another tool.")




In [ ]:
# Routing workflow
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

from typing_extensions import Literal
from langchain_core.messages import HumanMessage, SystemMessage

# Schema for structured output
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

model = init_chat_model("gpt-4", model_provider="openai")


# Schema for structured output to use as routing logic
class Route(BaseModel):
    step: Literal[ "variable_selector", "report_creator"] = Field(
        None, description="The next step in the routing process"
    )


# Augment the model with schema for structured output
router = model.with_structured_output(Route)


# State
class State(TypedDict):
    input: str
    decision: str
    output: str


# Nodes
# def query_cleaner(state: State):
#     """This function cleans the user query for confirmation by user"""
    
#     # TODO: crear interna
#     #result = model.invoke(state["input"])
#     return {"output": result.content}


# def variable_selector(state: State):
#     """This function will select the relevant variables based on the user query"""

#     # TODO: crear interna
#     # result = model.invoke(state["input"])
#     return {"output": result.content}

# variable_selector:
def variable_selector(state: State):
    """
    Selects the top variables based on their relevance to the user query.
    Returns:
        dict: A dictionary of selected variables with their grades.
    """
    # Call the internal function to get the grade dictionary
    tmp_pre_res_dict, tmp_grade_dict= _variable_selector(user_query, top_vals=30)

    return {"output": {'tmp_pre_res_dict': tmp_pre_res_dict, 'tmp_grade_dict': tmp_grade_dict}}


def report_creator(state: State):
    """This function will create a report"""

    # TODO: crear interna
    # result = model.invoke(state["input"])

    tmp_preproc_dic, final_smry_dict, structured_expert_results = _deep_analyzer(
        tmp_pre_res_dict=state["output"]['tmp_pre_res_dict'],
        tmp_grade_dict=state["output"]['tmp_grade_dict']
    )

    _report_writer(
        user_query=user_query,
        final_smry_dict=final_smry_dict,
        structured_expert_results=structured_expert_results,
        df_tables=tst_na_dict,  # Assuming tst_na_dict is defined globally
        pregs_dict=pregs_dict,  # Assuming pregs_dict is defined globally
        ruta_rep="reports/",
        write_pdf=True
    )


    return {"output": "Report created successfully."}


tmp_topic_st = ', '.join([st.replace('_', ' ').title() for st in enc_nom_dict.keys()])

def model_call_router(state: State):
    """Route the input to the appropriate node"""

    # Run the augmented model with structured output to serve as routing logic
    decision = router.invoke(
        [
            SystemMessage(
                content= f"""
                You are a useful assistant that guides users through a public opinion and survey analysis process. 
                You are fully bilingual in English and Spanish, and can do your work in either language, but will always reply in the languague used by the user.

                If the user asks which kinds of information you can provide, you will provide a numbered list of the general topics of the public opinion surveys you can work with: {tmp_topic_st}. 
                Only these topics are available and you will not list any others. However, you may encourage the user to ask you about any topic because the surveys touch upon a wide range of issues.  
                All surveys were all fielded in Mexico between 2014 and 2015. 
                All samples are probabilistic and were designed to be representative of the whole population of Mexico that is 15 years or older. 

                If the user has a specific query, you will decide the next step to take based on the user query.
                1. 'variable_selector' to select the relevant variables based on the user query if no variables have been selected yet. 
                2. 'report_creator' to create a report based on the user query once the variables have been selected. 
                You will return a JSON object with a single key 'step' that indicates the next step to take. The value of 'step' must be one of the following: 'variable_selector', 'report_creator'.
                """
            ),
            HumanMessage(content=state["input"]),
        ]
    )

    return {"decision": decision.step}


# Conditional edge function to route to the appropriate node
def route_decision(state: State):
    # Return the node name you want to visit next
    if state["decision"] == "variable_selector":
        return "variable_selector_fun"
    elif state["decision"] == "report_creator":
        return "report_creator_fun"


# Build workflow
router_builder = StateGraph(State)

# Add nodes
router_builder.add_node("variable_selector", variable_selector)
router_builder.add_node("variable_selector", report_creator)
# router_builder.add_node("model_call_3", model_call_3)
router_builder.add_node("model_call_router", model_call_router)

# Add edges to connect nodes
router_builder.add_edge(START, "model_call_router")
router_builder.add_conditional_edges(
    "model_call_router",
    route_decision,
    {  # Name returned by route_decision : Name of next node to visit
        "variable_selector": "variable_selector",
        "variable_selector": "variable_selector",
        # "model_call_3": "model_call_3",
    },
)
router_builder.add_edge("variable_selector_fun", END)
router_builder.add_edge("report_creator_fun", END)
# router_builder.add_edge("model_call_3", END)

# Compile workflow
router_workflow = router_builder.compile()

# Show the workflow
display(Image(router_workflow.get_graph().draw_mermaid_png()))

# Invoke
state = router_workflow.invoke({"input": user_query})
print(state["output"])

'Identidad Y Valores, Medio Ambiente, Pobreza, Cultura Politica, Religion Secularizacion Y Laicidad, Seguridad Publica, Salud, Indigenas, Sociedad De La Informacion, Envejecimiento, Derechos Humanos Discriminacion Y Grupos Vulnerables, Corrupcion Y Cultura De La Legalidad, La Condicion De Habitabilidad De Vivienda En Mexico, Globalizacion, Justicia, Juegos De Azar, Migracion, Federalismo, Genero, Cultura Constitucional, Cultura Lectura Y Deporte, Economia Y Empleo, Ninos Adolescentes Y Jovenes, Familia, Ciencia Y Tecnologia, Educacion'